# Two-Stage LoRA, Grouped Gaussian Readouts, OLMo2 and SmolLM3

This ensemble combines two-stage Qwen3 classifiers, a shared-prompt Gaussian readout, and separate OLMo2 and SmolLM3 members. The Qwen Gaussian method follows [kirilltym's version 4](https://www.kaggle.com/code/kirilltym/two-stage-lora-gaussian-readout?scriptVersionId=360050107), which scored 0.595. The two-stage adapters and n-gram ensemble come from [Nishant Dahal's version 2](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool?scriptVersionId=359887828), scored 0.592. The Gaussian implementation is adapted from [prvsiyan's The Prompt Is Not the Author](https://www.kaggle.com/code/prvsiyan/the-prompt-is-not-the-author?scriptVersionId=359792979).

The 8B model extracts test passages followed by all 2699 official training passages in one joint, length-sorted pass. A Gaussian head is fitted on the training representations and prompt IDs using within-prompt residual PCA128, Ledoit–Wolf covariance estimates and temperature 3.0. Its log probabilities enter at weight 0.10 only for the original text-derived groups of 2–18 passages. Pairs and triples use exact injective marginals; larger eligible groups use five cavity-EM iterations. The original SVC and Qwen plain/centered coefficients are retained.

The native24-layer OLMo2 classifier contributes at weight 0.03 before the original softmax and 100-step uniform balancing. This member previously scored 0.594 with the two-stage ensemble in [version 13](https://www.kaggle.com/code/lightningv08/0-594-two-stage-lora-olmo2-ensemble?scriptVersionId=360029464). That result does not establish its contribution to a new combination.

The full36-layer SmolLM3-3B classifier uses 300 public-pool warmup updates followed by 450 official-training updates. Its representation concatenates six first-token-masked mean pools, each passed through its corresponding trained LayerNorm once. The native plain head handles singleton and other ineligible passages. The centered head handles the same original Nishant groups of 2–18, subtracting each group's mean representation. A separate, pre-fitted PCA128 grouped Gaussian adds weight 0.10 within those groups. The complete Smol score then enters the ensemble at raw weight 0.03, without another member normalization or probability clipping. Both member class orders are explicitly remapped by label name before accumulation, in the order Qwen, OLMo2, SmolLM3.

Smol public pretraining improved the matched prompt-disjoint outer-split comparison, but a complete current-ensemble OOF comparison is unavailable. The fixed Gaussian component's conditional comparison had a confidence interval crossing zero. The final Smol weights and Gaussian parameters use all 2699 training rows and are not OOF artifacts. No leaderboard score is claimed for this combination.

The SVC and Qwen Gaussian heads are fitted at runtime; LoRA weights remain frozen and the additional Smol heads and Gaussian are pre-fitted. The Qwen process exits before two independent workers run OLMo2 on physical GPU0 and SmolLM3 on physical GPU1. Native Tesla T4 synthetic anchors guard feature and head replay. IDs and label names use Unicode arrays and no-pickle numeric transport, preserving leading zeros.

Credits: Nishant Dahal for the two-stage adapters, cleaned public pool and ensemble; kirilltym for the grouped Gaussian method on two-stage representations; prvsiyan for the Gaussian implementation; Yang Kuang Ou for style n-grams; Lucas Boesen for the LoRA recipe and centered twins; the Qwen team, Allen Institute for AI and HuggingFaceTB for the base models. Public pool attribution to Seyit Kaan Güneş, Will Rice, lxdlxdlxd, prvsiyan, alperen5252525, Hailey Cheng, kaplinskiy, Lucas Boesen, Abdullah Afify, Do Phuc Hung, ziyuzhou1207, keshavatearth and Michael Semenoff is preserved in the attached notices.

The [public pool and two-stage Qwen checkpoints](https://www.kaggle.com/datasets/nishantdahal/ntl-two-stage-lora) retain CC BY-SA 4.0 notices. Nishant and prvsiyan notebook sources, OLMo2 and SmolLM3 base models retain Apache 2.0 notices. The [Smol inference assets](https://www.kaggle.com/datasets/lightningv08/ntl-olmo2-native24-final-inference-assets?select=ntl-olmo2-native24-final-inference-assets?select=ntl-smollm3-two-stage-final-inference-assets) include model-specific readout provenance, license notices and public synthetic replay anchors. The required offline Smol base is [SmolLM3-3B-Base offline weights](https://www.kaggle.com/code/lightningv08/smollm3-3b-base-offline-weights).


In [ ]:
import hashlib,os
from pathlib import Path
assert os.name!='nt' and Path('/kaggle/input').is_dir(), 'This notebook uses attached offline Kaggle inputs.'
ROOT=Path('/kaggle/working/gaussian-two-stage-olmo-smol-source')
ROOT.mkdir(exist_ok=False)
FILES={'baseline_source.py': '"""Nishant v2 partition plus independently written grouped Gaussian8B integration."""\nimport ast, hashlib, json\nfrom pathlib import Path\n\ndef source_parts(here, query_csv=None):\n    raw=(Path(here)/\'nishant_0592_original.ipynb\').read_bytes()\n    assert hashlib.sha256(raw).hexdigest()==\'12e1e65bbc48a2839842d439fd7ae3846b9bf0d9913c4cdc308ebc698974b3fb\'\n    notebook=json.loads(raw)\n    code=\'\\n\\n\'.join(c[\'source\'] if isinstance(c[\'source\'],str) else \'\'.join(c[\'source\']) for c in notebook[\'cells\'] if c[\'cell_type\']==\'code\')\n    marker=\'P = softmax(LOGIT)\'\n    assert code.count(marker)==1\n    prefix,tail=code.split(marker)\n    original=\'Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])\'\n    assert prefix.count(original)==1\n    patch=(Path(here)/\'gaussian_integration.txt\').read_text(encoding=\'utf-8\').rstrip()\n    prefix=prefix.replace(original,patch)\n    marker_loop=\'for ckpt_name, mc in LORAS.items():\'\n    assert prefix.count(marker_loop)==1\n    prefix=prefix.replace(marker_loop,\'from gaussian_core import fit, scores\\n\\n\'+marker_loop)\n    if query_csv is not None:\n        needle=\'test = pd.read_csv(find("test.csv"))\'\n        assert prefix.count(needle)==1\n        prefix=prefix.replace(needle,\'test = pd.read_csv(\'+repr(str(query_csv))+\')\')\n    checkpoint_loop=\'    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):\\n\'\n    assert prefix.count(checkpoint_loop)==1\n    prefix=prefix.replace(checkpoint_loop,checkpoint_loop+"        directory = os.path.dirname(cfg)\\n        if not any(os.path.isfile(os.path.join(directory, name)) for name in (\'model.safetensors\', \'model.safetensors.index.json\', \'pytorch_model.bin\', \'pytorch_model.bin.index.json\')):\\n            continue  # metadata-only training copies are not checkpoints\\n")\n    functions={n.name:ast.get_source_segment(code,n) for n in ast.parse(code).body if isinstance(n,ast.FunctionDef)}\n    return prefix,marker+tail,functions\n', 'numeric_transport.py': '"""No-pickle numeric transport; validate the bytes actually written."""\nfrom pathlib import Path\nimport numpy as np\n\ndef save_numeric(path, **arrays):\n    arrays={key:np.asarray(value) for key,value in arrays.items()}\n    assert all(not value.dtype.hasobject for value in arrays.values()), \'Object arrays forbidden\'\n    assert all(value.dtype.kind in \'USbiuf\' for value in arrays.values())\n    assert all(np.isfinite(value).all() for value in arrays.values() if value.dtype.kind in \'f\')\n    path=Path(path)\n    assert not path.exists(), \'No automatic overwrite or reuse\'\n    np.savez_compressed(path,**arrays)\n    recovered=load_numeric(path)\n    assert set(recovered)==set(arrays)\n    for key,value in arrays.items():\n        assert recovered[key].dtype==value.dtype\n        np.testing.assert_array_equal(recovered[key],value)\n\ndef load_numeric(path):\n    with np.load(path,allow_pickle=False) as saved:\n        arrays={key:saved[key].copy() for key in saved.files}\n    assert all(not value.dtype.hasobject for value in arrays.values())\n    return arrays\n', 'run_baseline.py': '"""Two-stage Qwen and grouped Gaussian pass; retain numeric transport only."""\nimport json,os\nfrom pathlib import Path\nfrom baseline_source import source_parts\nfrom numeric_transport import save_numeric\n\nHERE=Path(__file__).resolve().parent\nOUT=Path(\'/kaggle/working\')\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\nassert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nquery=os.environ.get(\'NTL_QUERY_CSV\')\nimport numpy as np\nimport pandas as pd\n# Identity failures are detected before any model is loaded.\npaths=[Path(query)] if query is not None else [p/\'test.csv\' for p in\n    (Path(\'/kaggle/input/competitions/name-that-llm\'),Path(\'/kaggle/input/name-that-llm\')) if (p/\'test.csv\').is_file()]\nassert len(paths)==1\ncanonical=pd.read_csv(paths[0],usecols=[\'id\'],dtype=str)\nassert not canonical[\'id\'].isna().any()\ncanonical_ids=np.asarray(canonical[\'id\'].tolist(),dtype=str)\nassert canonical_ids.dtype.kind==\'U\' and len(np.unique(canonical_ids))==len(canonical_ids)>0\nprefix,tail,functions=source_parts(HERE,query)\nenv={\'__name__\':\'__main__\',\'__file__\':str(HERE/\'nishant_0592_original.ipynb\')}\nexec(compile(prefix,\'<two-stage-plus-grouped-Gaussian-before-final-softmax>\',\'exec\'),env)\nnp=env[\'np\']\n# Canonical identity was checked before the heavy source was executed.\nids=canonical_ids\nassert len(ids)==len(env[\'test\'])\nlabels=np.asarray(env[\'LABELS\'],dtype=str)\nraw=env[\'LOGIT\']\nassert raw.dtype==np.float64 and raw.shape==(len(ids),18) and np.isfinite(raw).all()\nassert len(set(ids.tolist()))==len(ids) and set(labels)==set(json.loads((HERE/\'olmo-spec.json\').read_text())[\'classes\'])\nassert env[\'W_SVC\']==1.021 and env[\'TWIN\']==dict(thresh=.85,min_size=2,max_size=18)\nassert env[\'LORAS\'][\'lora8b_v1\'][\'w\']==.226 and env[\'LORAS\'][\'lora8b_v1\'][\'twin_w\']==.079\nassert env[\'LORAS\'][\'lora14b_v1\'][\'w\']==.429 and env[\'LORAS\'][\'lora14b_v1\'][\'twin_w\']==.150\narrays=dict(raw=raw,ids=ids,labels=labels)\nsave_numeric(cache/\'baseline-state.npz\',**arrays)\nsave_numeric(OUT/\'two_stage_baseline_recovery.npz\',**arrays)\nmeta=dict(source_notebook_id=139520703,version=2,script_version_id=359887828,grouped_Gaussian_method_notebook_id=138904218,grouped_Gaussian_method_version=4,grouped_Gaussian_method_script_version_id=360050107,rows=len(ids),baseline_start_unix=env[\'T0\'],\n    all_original_weights=True,heavy_baseline_passes=1,query_csv_override=query is not None,train_model_forward_passes=1,train_8B_rows=2699,Gaussian_fit=True,Gaussian_dimension=128,Gaussian_temperature=3.,Gaussian_weight=.10,Gaussian_group_sizes=[2,18],Gaussian_core_author="prvsiyan",new_Gaussian_head_OOF_available=False)\n(cache/\'baseline-meta.json\').write_text(json.dumps(meta,indent=2))\n(OUT/\'two_stage_baseline_audit.json\').write_text(json.dumps(meta,indent=2))\nprint(\'TWO_STAGE_RAW_COMPLETE_EXITING_TO_RELEASE_BOTH_GPUS\',flush=True)\n', 'run_baseline_with_groups.py': '"""Run the frozen v14 prefix, then export its ACTUAL original Nishant groups."""\nimport os,runpy\nfrom pathlib import Path\nimport numpy as np\nfrom numeric_transport import save_numeric\nHERE=Path(__file__).resolve().parent\nscope=runpy.run_path(str(HERE/\'run_baseline.py\'),run_name=\'__main__\')\nenv=scope[\'env\'];ids=scope[\'canonical_ids\']\nassert env[\'TWIN\']==dict(thresh=.85,min_size=2,max_size=18)\ngroups=np.asarray(env[\'groups\'],dtype=np.int64)\nsize=np.asarray(env[\'size\'],dtype=np.int64)\neligible=np.asarray(env[\'twin_rows\'],dtype=bool)\nassert groups.shape==size.shape==eligible.shape==ids.shape\nnp.testing.assert_array_equal(size,np.bincount(groups)[groups])\nnp.testing.assert_array_equal(eligible,(size>=2)&(size<=18))\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\nassert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nsave_numeric(cache/\'baseline-groups.npz\',ids=ids,groups=groups,size=size,eligible=eligible)\nprint(\'ORIGINAL_NISHANT_GROUPS_EXPORTED_NO_RECOMPUTATION\',flush=True)\n', 'compute_olmo.py': '"""Frozen native24 OLMo2 plain readout; one query pass, no fitting."""\nimport hashlib,json,os,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom olmo_component import OLMoFinalPlain\n\nhere=Path(__file__).resolve().parent;config=json.loads((here/\'candidate-config.json\').read_text())\nassert config[\'source_ready\'] and config[\'OLMo_plain_raw_coefficient\']==.03\nspec=json.loads((here/\'olmo-spec.json\').read_text())\nassert spec[\'inference_authorized\'] and spec[\'root_production_admitted\'] and spec[\'public_asset_release_audited\']\ndef sha(p):\n    with p.open(\'rb\') as f:return hashlib.file_digest(f,\'sha256\').hexdigest()\nassets=[p for p in Path(\'/kaggle/input\').rglob(\'asset_manifest.json\') if sha(p)==spec[\'compact_manifest_sha256\']]\nbase_config=next(r for r in spec[\'base_files\'] if r[\'name\']==\'config.json\')\nbases=[p for p in Path(\'/kaggle/input\').rglob(\'config.json\') if sha(p)==base_config[\'sha256\']]\n\ntests=[p/\'test.csv\' for p in (Path(\'/kaggle/input/competitions/name-that-llm\'),Path(\'/kaggle/input/name-that-llm\')) if (p/\'test.csv\').is_file()]\nif os.environ.get(\'NTL_QUERY_CSV\'): tests=[Path(os.environ[\'NTL_QUERY_CSV\'])]\nassert len(assets)==len(bases)==len(tests)==1\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve();assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nassert not (cache/\'olmo_plain.npz\').exists(),\'No automatic repeat/reuse\'\nquery=pd.read_csv(tests[0],usecols=[\'id\',\'text\'],dtype=str)\nquery[\'text\']=query[\'text\'].fillna(\'\').astype(str)\nids=query.id.to_numpy(dtype=str);labels=np.asarray(spec[\'classes\'],dtype=str)\nnp.testing.assert_array_equal(query.id.to_numpy(dtype=str),ids)\nassert os.environ[\'CUDA_VISIBLE_DEVICES\'].startswith(\'GPU-\')\nstarted=time.monotonic()\nmodel=OLMoFinalPlain(here/\'olmo-spec.json\',assets[0].parent,bases[0].parent,None)\nnp.testing.assert_array_equal(model.readouts[\'classes\'],labels)\nfrom worker_barrier import ready_and_wait\nready_and_wait(\'olmo\')\nimport torch\nassert torch.cuda.device_count()==1\nbegin,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\nbegin.record();result=model.predict_plain_logp(ids,query.text.tolist());end.record();torch.cuda.synchronize()\nevent_ms=float(begin.elapsed_time(end));assert event_ms>0\nnp.testing.assert_array_equal(result[\'ids\'],ids);np.testing.assert_array_equal(result[\'classes\'],labels)\nassert result[\'plain_logp\'].dtype==np.float32 and result[\'plain_logp\'].shape==(len(ids),18) and np.isfinite(result[\'plain_logp\']).all()\nassert torch.cuda.max_memory_reserved()<=13*1024**3\nfrom numeric_transport import save_numeric\nsave_numeric(cache/\'olmo_plain.npz\',ids=ids,labels=labels,plain_logp=result[\'plain_logp\'])\nsave_numeric(Path(\'/kaggle/working/olmo_plain_recovery.npz\'),ids=ids,labels=labels,plain_logp=result[\'plain_logp\'])\nPath(\'/kaggle/working/olmo_inference_audit.json\').write_text(json.dumps(dict(status=\'FROZEN_OLMO2_NATIVE24_PLAIN_QUERY_COMPLETE\',rows=len(ids),feature_width=24576,plain_dtype=\'float32\',classes=labels.tolist(),original_model_first24_final_RMSNorm=True,pool_layers=[4,8,12,16,20,24],synthetic_anchor_replay=model.anchor_replay,CUDA_VISIBLE_DEVICES=os.environ[\'CUDA_VISIBLE_DEVICES\'],query_cuda_event_ms=event_ms,peak_reserved_bytes=torch.cuda.max_memory_reserved(),wall_seconds=time.monotonic()-started,model_fits=0),indent=2))\n', 'finish_prediction.py': '"""Add one owned FP32 term, then execute the original full Sinkhorn tail."""\nimport json,os,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom baseline_source import source_parts\nfrom numeric_transport import load_numeric,save_numeric\n\nHERE=Path(__file__).resolve().parent\n\ndef finish(data,own,output_dir,coefficient=.03,smol=None,smol_coefficient=.03):\n    output_dir=Path(output_dir)\n    assert coefficient in (0.,.03)\n    assert set(data)=={\'ids\',\'labels\',\'raw\'} and set(own)=={\'ids\',\'labels\',\'plain_logp\'}\n    ids,labels,raw=data[\'ids\'],data[\'labels\'],data[\'raw\']\n    assert ids.dtype.kind==\'U\' and labels.dtype.kind==\'U\' and len(np.unique(ids))==len(ids)\n    assert raw.dtype==np.float64 and raw.shape==(len(ids),18) and np.isfinite(raw).all()\n    np.testing.assert_array_equal(own[\'ids\'],ids)\n    assert own[\'labels\'].dtype.kind==\'U\' and len(set(own[\'labels\'].tolist()))==18\n    assert set(own[\'labels\'].tolist())==set(labels.tolist())\n    reorder=np.asarray([own[\'labels\'].tolist().index(label) for label in labels],dtype=int)\n    original=own[\'plain_logp\']\n    assert original.dtype==np.float32 and original.shape==raw.shape and np.isfinite(original).all()\n    ordered=original[:,reorder]\n    # Match the established owned-member FP32 exp/clip/log contribution exactly.\n    term=coefficient*np.log(np.clip(np.exp(ordered),1e-9,1.))\n    assert term.dtype==np.float32 and np.isfinite(term).all()\n    LOGIT=raw.copy()\n    LOGIT+=term\n    np.testing.assert_array_equal(LOGIT,raw+term)\n    assert smol_coefficient in (0.,.03)\n    if smol is not None:\n        assert set(smol)=={\'ids\',\'labels\',\'score\'}\n        np.testing.assert_array_equal(smol[\'ids\'],ids)\n        assert smol[\'labels\'].dtype.kind==\'U\' and len(set(smol[\'labels\']))==18\n        assert set(smol[\'labels\'])==set(labels)\n        smol_order=np.asarray([smol[\'labels\'].tolist().index(label) for label in labels],dtype=int)\n        S=smol[\'score\']\n        assert S.dtype==np.float64 and S.shape==raw.shape and np.isfinite(S).all()\n        # Preserve parent addorder exactly: original Qwen/Gaussian -> OLMo03 -> Smol03.\n        # S already contains routed FP32 logp + eligible-only .10 FP64 Gaussian.\n        # Never normalize, exponentiate or clip this member again.\n        smol_term=smol_coefficient*S[:,smol_order]\n        LOGIT+=smol_term\n        np.testing.assert_array_equal(LOGIT,(raw+term)+smol_term)\n    prefix,tail,functions=source_parts(HERE)\n    env=dict(np=np,pd=pd,time=time,T0=time.time(),LOGIT=LOGIT,LABELS=labels.tolist(),K=18,test=pd.DataFrame({\'id\':ids}))\n    exec(compile(functions[\'softmax\'],\'<original-nishant-softmax>\',\'exec\'),env)\n    # Change only the output pathname, preserving every mathematical operation.\n    needle=\'sub.to_parquet("submission.parquet", index=False)\'\n    assert tail.count(needle)==1\n    tail=tail.replace(needle,\'sub.to_parquet(\'+repr(str(output_dir/\'submission.parquet\'))+\', index=False)\')\n    exec(compile(tail,\'<exact-nishant-v2-full-100-step-sinkhorn-tail>\',\'exec\'),env)\n    observed=pd.read_parquet(output_dir/\'submission.parquet\')\n    np.testing.assert_array_equal(observed.id.to_numpy(dtype=str),ids)\n    np.testing.assert_array_equal(observed.model.to_numpy(dtype=str),env[\'sub\'].model.to_numpy(dtype=str))\n    assert observed.columns.tolist()==[\'id\',\'model\'] and len(observed)==len(ids)\n    return env\n\ndef main():\n    cache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\n    assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\n    data=load_numeric(cache/\'baseline-state.npz\')\n    own=load_numeric(cache/\'olmo_plain.npz\')\n    smol=load_numeric(cache/\'smol_score.npz\')\n    out=Path(\'/kaggle/working\')\n    candidate=finish(data,own,out,.03,smol,.03)\n    save_numeric(out/\'two_stage_final_scores.npz\',ids=data[\'ids\'],labels=data[\'labels\'],probabilities=candidate[\'P\'])\n    audit=dict(status=\'TWO_STAGE_GAUSSIAN_OLMO03_SMOL03_COMPLETE\',rows=len(data[\'ids\']),\n        OLMo_raw_weight=.03,Smol_raw_weight=.03,Smol_Gaussian_weight=.10,\n        Smol_member_renormalized=False,original_uniform_Sinkhorn100=True,\n        original_Nishant_group_sizes=[2,18],own_class_name_remaps=2,\n        deterministic_addorder=[\'original_Qwen_Gaussian\',\'OLMo03\',\'Smol03\'],\n        full_stack_OOF_available=False,new_candidate_score=None)\n    (out/\'candidate_audit.json\').write_text(json.dumps(audit,indent=2))\n\nif __name__==\'__main__\':main()\n', 'run_candidate.py': '"""Frozen baseline exits, then UUID-isolated OLMo0+Smol1 with actual telemetry."""\nimport hashlib,importlib.metadata,json,os,signal,subprocess,sys,tempfile,time\nfrom pathlib import Path\nHERE=Path(__file__).resolve().parent\nC=json.loads((HERE/\'candidate-config.json\').read_text())\nassert os.name!=\'nt\' and Path(\'/kaggle/input\').is_dir() and C[\'source_ready\']\nmode=os.environ.get(\'NTL_RUN_MODE\',\'candidate\');assert mode in (\'candidate\',\'diagnostic\')\nif mode==\'candidate\':assert C[\'candidate_admitted\'],\'CLOSED draft; root admission required before candidate dispatch\'\nelse:assert C[\'cloud_probe_authorized\'] and not C[\'candidate_admitted\'],\'CLOSED targeted diagnostic; separate root ACK required\'\nassert C[\'Smol_asset_publication_admitted\'],\'Required final Smol dataset not yet published\'\nfor name,digest in C[\'source_sha256\'].items():assert hashlib.sha256((HERE/name).read_bytes()).hexdigest()==digest,name\nS=json.loads((HERE/\'smol-spec.json\').read_text())\nassert S[\'public_asset_release_admitted\'] and S[\'asset_ref\']\nassert {n:importlib.metadata.version(n) for n in S[\'core_versions\']}==S[\'core_versions\']\nassert not os.environ.get(\'NTL_QUERY_CSV\') and not os.environ.get(\'NTL_SMOKE\')\nmapping=[]\nfor line in subprocess.check_output([\'nvidia-smi\',\'--query-gpu=index,uuid,name\',\'--format=csv,noheader,nounits\'],text=True).strip().splitlines():\n    index,uuid,name=[x.strip() for x in line.split(\',\',2)]\n    mapping.append(dict(index=int(index),uuid=uuid,name=name))\nmapping.sort(key=lambda x:x[\'index\'])\nassert [x[\'index\'] for x in mapping]==[0,1] and len({x[\'uuid\'] for x in mapping})==2 and all(\'T4\' in x[\'name\'] for x in mapping)\nPath(\'/kaggle/temp\').mkdir(exist_ok=True)\ncache=Path(tempfile.mkdtemp(prefix=\'ntl-qwen-olmo-smol-\',dir=\'/kaggle/temp\'))\nenv=os.environ.copy();env[\'NTL_CANDIDATE_TEMP\']=str(cache)\nif mode==\'diagnostic\':env[\'NTL_QUERY_CSV\']=str(cache/\'diagnostic60-query.csv\')\nenv[\'HF_HUB_OFFLINE\']=env[\'TRANSFORMERS_OFFLINE\']=\'1\';env[\'TOKENIZERS_PARALLELISM\']=\'false\'\nfor n in [\'OMP_NUM_THREADS\',\'MKL_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\']:env[n]=\'2\'\n# Leave 30s for bounded child termination before the notebook\'s external 900s cap.\ndeadline=time.monotonic()+(870 if mode==\'diagnostic\' else 10800)\n\ndef remaining():\n    seconds=deadline-time.monotonic()\n    assert seconds>0,\'Bounded total runtime; no retry\'\n    return seconds\n\ndef stop(children):\n    for child in children:\n        if child.poll() is None:os.killpg(child.pid,signal.SIGTERM)\n    for child in children:\n        try:child.wait(timeout=5)\n        except subprocess.TimeoutExpired:os.killpg(child.pid,signal.SIGKILL);child.wait()\n\ndef devices():\n    gpu=subprocess.check_output([\'nvidia-smi\',\'--query-gpu=index,uuid,utilization.gpu,memory.used\',\'--format=csv,noheader,nounits\'],text=True,timeout=10)\n    apps=subprocess.check_output([\'nvidia-smi\',\'--query-compute-apps=pid,gpu_uuid\',\'--format=csv,noheader,nounits\'],text=True,timeout=10)\n    values=[]\n    for line in gpu.strip().splitlines():\n        index,uuid,util,memory=[x.strip() for x in line.split(\',\')]\n        values.append(dict(index=int(index),uuid=uuid,util=float(util),memory_MiB=float(memory)))\n    values.sort(key=lambda x:x[\'index\'])\n    assert [x[\'uuid\'] for x in values]==[x[\'uuid\'] for x in mapping]\n    pairs={tuple(x.strip() for x in line.split(\',\')[:2]) for line in apps.strip().splitlines() if line.strip()}\n    return values,pairs,apps\n\nreports=[];baseline_env=env.copy();baseline_env[\'CUDA_VISIBLE_DEVICES\']=\',\'.join(x[\'uuid\'] for x in mapping)\nwith Path(\'/kaggle/working/baseline.log\').open(\'w\',buffering=1) as log:\n    start=time.monotonic()\n    baseline_script=\'run_synthetic_baseline.py\' if mode==\'diagnostic\' else \'run_baseline_with_groups.py\'\n    baseline=subprocess.Popen([sys.executable,\'-u\',str(HERE/baseline_script)],env=baseline_env,cwd=\'/kaggle/working\',stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\n    try:\n        while baseline.poll() is None:\n            remaining();time.sleep(.5)\n        assert baseline.returncode==0,\'Baseline failed; no retry\'\n    finally:stop([baseline])\n    reports.append(dict(stage=\'baseline\',PID=baseline.pid,seconds=time.monotonic()-start,exit_code=0))\nassert (cache/\'baseline-state.npz\').is_file() and (cache/\'baseline-groups.npz\').is_file()\nfrom numeric_transport import load_numeric\nbaseline_rows=int(len(load_numeric(cache/\'baseline-state.npz\')[\'ids\']))\nif mode==\'diagnostic\':assert baseline_rows==60\n# Process termination, not a Python del, releases every baseline CUDA allocation.\nvalues,pairs,apps=devices()\nassert not pairs,(\'Baseline exited but a CUDA process remains\',apps)\nchildren=[];logs=[];seen=set();overlap=0;started=time.monotonic()\ntry:\n    for i,name in enumerate([\'olmo\',\'smol\']):\n        child_env=env.copy();child_env[\'CUDA_VISIBLE_DEVICES\']=mapping[i][\'uuid\']\n        log=Path(\'/kaggle/working\',name+\'.log\').open(\'w\',buffering=1);logs.append(log)\n        children.append(subprocess.Popen([sys.executable,\'-u\',str(HERE/(\'compute_\'+name+\'.py\'))],env=child_env,cwd=\'/kaggle/working\',stdout=log,stderr=subprocess.STDOUT,start_new_session=True))\n    while True:\n        remaining();codes=[c.poll() for c in children]\n        assert all(c in (None,0) for c in codes),(\'Owned worker failed; no retry\',codes)\n        ready=[cache/(name+\'-READY.json\') for name in [\'olmo\',\'smol\']]\n        if all(p.exists() for p in ready) and not (cache/\'OWNED-GO\').exists():\n            for i,p in enumerate(ready):\n                r=json.loads(p.read_text());assert r[\'PID\']==children[i].pid and r[\'UUID\']==mapping[i][\'uuid\']\n            (cache/\'OWNED-GO\').touch()\n        values,pairs,apps=devices()\n        matched=[(str(c.pid),mapping[i][\'uuid\']) in pairs for i,c in enumerate(children)]\n        for i,v in enumerate(matched):\n            if v:seen.add(i)\n        active=(cache/\'OWNED-GO\').exists() and all(matched) and all(v[\'util\']>0 and v[\'memory_MiB\']>0 for v in values)\n        overlap+=int(active)\n        with Path(\'/kaggle/working/owned_gpu_telemetry.jsonl\').open(\'a\') as f:\n            f.write(json.dumps(dict(seconds=time.monotonic()-started,PID=[c.pid for c in children],physical_devices=values,processes=apps,matched=matched,after_GO=(cache/\'OWNED-GO\').exists(),simultaneous_PID_UUID_util=active))+\'\\n\')\n        if all(c==0 for c in codes):break\n        time.sleep(.25)\n    assert (cache/\'olmo_plain.npz\').is_file() and (cache/\'smol_score.npz\').is_file()\n    from concurrency_evidence import assess_owned_evidence\n    evidence=assess_owned_evidence(baseline_rows,seen,overlap,\n        [json.loads((cache/(name+\'-READY.json\')).read_text()) for name in [\'olmo\',\'smol\']],\n        [json.loads(Path(\'/kaggle/working\',name+\'_inference_audit.json\').read_text()) for name in [\'olmo\',\'smol\']],\n        [c.pid for c in children],mapping)\nfinally:\n    stop(children)\n    for log in logs:log.close()\nvalues,pairs,apps=devices();assert not pairs,(\'Owned CUDA child survives exit\',apps)\nsubprocess.run([sys.executable,\'-u\',str(HERE/\'finish_prediction.py\')],env=env,cwd=\'/kaggle/working\',check=True,timeout=min(120,remaining()))\nassert json.loads(Path(\'/kaggle/working/candidate_audit.json\').read_text())[\'status\']==\'TWO_STAGE_GAUSSIAN_OLMO03_SMOL03_COMPLETE\'\nPath(\'/kaggle/working/parallel_execution_audit.json\').write_text(json.dumps(dict(status=\'COMPLETE_BASELINE_THEN_TWO_OWNED_WORKERS\',mode=mode,baseline=reports[0],baseline_is_synthetic_stub=mode==\'diagnostic\',own_worker_PIDs=[c.pid for c in children],GPU_mapping=mapping,evidence=evidence,baseline_owned_overlap=False,parent_model_initializations=0,automatic_retries=0,addorder=[\'original_Qwen_Gaussian\',\'OLMo03\',\'Smol03\']),indent=2))\n', 'olmo_component.py': '"""Portable trained final450 OLMo2 native first24 plain readout; no fitting."""\nimport hashlib,importlib.metadata,json,os\nfrom pathlib import Path\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:return hashlib.file_digest(stream,\'sha256\').hexdigest()\ndef validate_contract(spec,artifact_root,base_root,diagnostic_only=False):\n    assert not diagnostic_only and spec[\'inference_authorized\'] and spec[\'root_production_admitted\']\n    root,base=Path(artifact_root),Path(base_root)\n    assert sha(root/\'asset_manifest.json\')==spec[\'compact_manifest_sha256\']\n    manifest=json.loads((root/\'asset_manifest.json\').read_text())\n    assert manifest[\'status\']==\'PUBLIC_SAFE_OLMO2_NATIVE24_FINAL_INFERENCE\'\n    for flag in (\'contains_canonical_competition_IDs\',\'contains_competition_text_or_token_IDs\',\'contains_targets\',\'contains_optimizer_or_RNG\',\'contains_training_or_query_features\'):\n        assert manifest[flag] is False\n    assert {r[\'relative_path\'] for r in manifest[\'files\']}==set(spec[\'public_file_allowlist\'])\n    for rec in manifest[\'files\']:\n        path=root/rec[\'relative_path\'];assert path.resolve().is_relative_to(root.resolve())\n        assert path.stat().st_size==rec[\'bytes\'] and sha(path)==rec[\'sha256\']\n    provenance=json.loads((root/\'provenance.json\').read_text())\n    assert provenance[\'source_notebook_id\']==139418845 and provenance[\'source_script_version_id\']==359697707\n    assert provenance[\'source_worker_sha256\']==spec[\'training_worker_sha256\']\n    assert provenance[\'base_revision\']==spec[\'base_revision\'] and provenance[\'native_final_RMSNorm_retained\']\n    assert provenance[\'retained_layers\']==24 and provenance[\'pool_layers\']==[4,8,12,16,20,24] and provenance[\'feature_width\']==24576\n    assert provenance[\'successful_updates\']==450 and provenance[\'epochs\']==3 and provenance[\'final_not_OOF\']\n    assert provenance[\'final_overflow_attempts\']==0 and provenance[\'outer0_bounded_AMP_recoveries\']==4\n    assert provenance[\'classes\']==spec[\'classes\'] and provenance[\'synthetic_texts\']==spec[\'synthetic_texts\']\n    assert provenance[\'original_private_anchor_replay_passed\'] and provenance[\'actual_GPU_diagnostic_passed\']\n    config=json.loads((root/\'adapter/adapter_config.json\').read_text())\n    assert config[\'r\']==8 and config[\'lora_alpha\']==16 and config[\'lora_dropout\']==.05 and config[\'bias\']==\'none\'\n    assert set(config[\'target_modules\'])=={\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'}\n    assert config[\'auto_mapping\'][\'base_model_class\']==\'Olmo2Model\' and config[\'peft_type\']==\'LORA\'\n    for rec in spec[\'base_files\']:\n        path=base/rec[\'name\'];assert path.stat().st_size==rec[\'bytes\'] and sha(path)==rec[\'sha256\']\n    return root,base,provenance\nclass OLMoFinalPlain:\n    def __init__(self, spec_path, artifact_root, base_root, anchor_train_csv, diagnostic_only=False):\n        self.spec = json.loads(Path(spec_path).read_text())\n        for name, digest in self.spec["source_hashes"].items():\n            assert sha(Path(__file__).parent / name) == digest\n        checkpoint, base_dir, report = validate_contract(self.spec, artifact_root, base_root, diagnostic_only=diagnostic_only)\n        assert os.name != "nt" and Path("/kaggle/input").is_dir(), "Remote-only component"\n        os.environ["HF_HUB_OFFLINE"] = "1"\n        os.environ["TRANSFORMERS_OFFLINE"] = "1"\n        os.environ["TOKENIZERS_PARALLELISM"] = "false"\n        import importlib.util\n        finder = importlib.util.find_spec\n        importlib.util.find_spec = lambda name, *a, **kw: None if name.split(".")[0] == "torchao" else finder(name, *a, **kw)\n        # Keep the successful FULL worker\'s torchao mask active through PEFT dispatch.\n        import torch\n        from torch import nn\n        from peft import PeftModel\n        import peft.import_utils as peft_imports\n        peft_imports.is_torchao_available = lambda *a, **kw: False\n        import numpy as np\n        import pandas as pd\n        from safetensors.torch import load_file\n        from scipy.special import log_softmax, softmax\n        from olmo_owned_path import load_owned_base, make_representation, tokenize_owned\n\n        assert torch.cuda.is_available() and torch.cuda.get_device_capability(0) == (7, 5)\n        assert {name: importlib.metadata.version(name) for name in self.spec["core_versions"]} == self.spec["core_versions"]\n        torch.set_num_threads(2)\n        loading = []\n        # Fail before LM allocation on actual serialized readout geometry/class mismatch.\n        with np.load(checkpoint/\'readouts.npz\',allow_pickle=False) as checked:\n            np.testing.assert_array_equal(checked[\'classes\'],np.asarray(self.spec[\'classes\'],dtype=str))\n            np.testing.assert_array_equal(checked[\'pool_layers\'],[4,8,12,16,20,24])\n            for route in (\'plain\',\'center\'):\n                assert checked[route+\'_coef\'].shape==(18,24576) and checked[route+\'_coef\'].dtype==np.float32\n                assert checked[route+\'_intercept\'].shape==(18,) and checked[route+\'_intercept\'].dtype==np.float32\n        base = load_owned_base(base_dir, loading, lambda: None)\n        # Semantic resolution of actual serialized7suffixes; never assert serialized168 paths.\n        native_targets={name:module for name,module in base.named_modules() if isinstance(module,nn.Linear) and name.startswith(\'layers.\') and name.rsplit(\'.\',1)[-1] in {\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\',\'gate_proj\',\'up_proj\',\'down_proj\'}}\n        assert len(native_targets)==168\n        shapes={\'self_attn.q_proj\':(4096,4096),\'self_attn.k_proj\':(4096,4096),\'self_attn.v_proj\':(4096,4096),\'self_attn.o_proj\':(4096,4096),\'mlp.gate_proj\':(4096,11008),\'mlp.up_proj\':(4096,11008),\'mlp.down_proj\':(11008,4096)}\n        for layer in range(24):\n            for suffix,shape in shapes.items():\n                module=native_targets[f\'layers.{layer}.{suffix}\'];assert (module.in_features,module.out_features)==shape and module.bias is None\n        del native_targets,module\n        self.loading = loading\n        self.model = PeftModel.from_pretrained(base, checkpoint / "adapter", is_trainable=False, local_files_only=True).eval()\n        expanded=[name for name,module in self.model.named_modules() if hasattr(module,\'lora_A\')]\n        assert len(expanded)==168\n        assert len(self.model.base_model.model.layers)==24 and self.model.base_model.model.norm is not None\n        assert not any(parameter.requires_grad for parameter in self.model.parameters())\n        assert all(parameter.dtype == torch.float32 for name, parameter in self.model.named_parameters() if "lora_" in name)\n        self.layers = [4, 8, 12, 16, 20, 24]\n        self.norms = nn.ModuleList([nn.LayerNorm(4096) for _ in self.layers]).cuda().float()\n        saved = load_file(checkpoint / "heads.safetensors")\n        assert all(value.dtype == torch.float32 and bool(torch.isfinite(value).all()) for value in saved.values())\n        self.norms.load_state_dict({key.removeprefix("norms."): value for key, value in saved.items() if key.startswith("norms.")}, strict=True)\n        self.norms.eval()\n        self.norms.requires_grad_(False)\n        with np.load(checkpoint / "readouts.npz", allow_pickle=False) as readouts:\n            self.readouts = {key: readouts[key].copy() for key in readouts.files}\n        np.testing.assert_array_equal(self.readouts["classes"], np.asarray(self.spec["classes"], dtype=str))\n        np.testing.assert_array_equal(self.readouts["pool_layers"], self.layers)\n        assert self.readouts["plain_coef"].shape == self.readouts["center_coef"].shape == (18, 24576)\n        assert self.readouts["plain_intercept"].shape == self.readouts["center_intercept"].shape == (18,)\n        for route, prefix in (("plain", "plain"), ("center", "centered")):\n            for key, state_key in (("coef", "weight"), ("intercept", "bias")):\n                array = self.readouts[route + "_" + key]\n                assert array.dtype == np.float32 and np.isfinite(array).all()\n                np.testing.assert_array_equal(array, saved[prefix + "." + state_key].numpy())\n        # Synthetic literals only; canonical private B2 replay was checked before packaging.\n        self._diagnostic_only=False\n        self._torch,self._np=torch,np\n        self._make_representation,self._tokenize_owned=make_representation,tokenize_owned\n        self._log_softmax=log_softmax\n        with np.load(checkpoint/\'synthetic_anchors.npz\',allow_pickle=False) as anchors:\n            assert set(anchors.files)=={\'ids\',\'texts\',\'features\',\'plain_logp\',\'classes\'}\n            np.testing.assert_array_equal(anchors[\'ids\'],[\'synthetic-a\',\'synthetic-b\',\'synthetic-c\'])\n            np.testing.assert_array_equal(anchors[\'classes\'],self.readouts[\'classes\'])\n            texts=anchors[\'texts\'].tolist();assert texts==self.spec[\'synthetic_texts\']\n            expected=anchors[\'features\'].copy()\n            expected_logp=anchors[\'plain_logp\'].copy()\n            assert expected.shape==(3,24576) and expected.dtype==np.float32\n            assert expected_logp.shape==(3,18) and expected_logp.dtype==np.float32\n        self.tokenizer,_=tokenize_owned(base_dir,texts)\n        recorded_reloaded=expected\n        fresh = self._features(texts, sort_by_length=True)\n        fresh_logp=log_softmax(fresh @ self.readouts[\'plain_coef\'].T+self.readouts[\'plain_intercept\'],axis=1)\n        np.testing.assert_allclose(fresh_logp,expected_logp,rtol=0,atol=1e-6)\n        assert expected.dtype == fresh.dtype == np.float32 and np.isfinite(expected).all()\n        np.testing.assert_allclose(fresh, expected, rtol=1e-5, atol=1e-5)\n        np.testing.assert_allclose(fresh, recorded_reloaded, rtol=1e-5, atol=1e-5)\n        parity = {}\n        for route in ("plain", "center"):\n            a = expected if route == "plain" else expected - expected.mean(0)\n            b = fresh if route == "plain" else fresh - fresh.mean(0)\n            coef, intercept = self.readouts[route + "_coef"], self.readouts[route + "_intercept"]\n            before = softmax(log_softmax(a @ coef.T + intercept, axis=1), axis=1)\n            after = softmax(log_softmax(b @ coef.T + intercept, axis=1), axis=1)\n            parity[route] = float(np.max(np.abs(before - after)))\n            assert parity[route] <= 1e-6\n        self.anchor_replay = {"feature_maxabs": float(np.max(np.abs(fresh - expected))), "probability_maxabs": parity,\n                              "feature_rtol": 1e-5, "feature_atol": 1e-5, "probability_atol": 1e-6}\n\n    def _features(self, texts, sort_by_length=True):\n        torch, np = self._torch, self._np\n        tokenizer, tokens = self._tokenize_owned(None, texts, tokenizer=self.tokenizer)\n        representation = self._make_representation(self.model, tokenizer, tokens, self.layers, self.norms)\n        features = np.empty((len(tokens), 24576), dtype=np.float32)\n        order = np.argsort(np.asarray([len(row) for row in tokens]), kind="stable") if sort_by_length else np.arange(len(tokens))\n        with torch.inference_mode():\n            for start in range(0, len(order), 2):\n                positions = order[start:start + 2]\n                features[positions] = representation(positions)[0].cpu().numpy()\n        assert np.isfinite(features).all()\n        return features\n\n    def predict_plain_logp(self, ids, texts):\n        np = self._np\n        ids = np.asarray(ids, dtype=str)\n        assert ids.ndim == 1 and len(ids) == len(texts) > 0 and len(np.unique(ids)) == len(ids)\n        if self._diagnostic_only:\n            assert len(ids) in (3, 5) and ids.tolist() == self._diagnostic_ids[:len(ids)]\n            assert list(texts) == self._diagnostic_texts[:len(ids)], "Only fixed synthetic3/5-row diagnostic fixtures admitted"\n        features = self._features(texts)\n        logp = self._log_softmax(features @ self.readouts["plain_coef"].T + self.readouts["plain_intercept"], axis=1)\n        assert logp.shape == (len(ids), 18) and logp.dtype == np.float32 and np.isfinite(logp).all()\n        return {"ids": ids, "classes": self.readouts["classes"].copy(), "plain_logp": logp,\n                "features": features, "pool_layers": np.asarray(self.layers)}\n', 'olmo_owned_path.py': '"""Exact native final450 Olmo2Model prefix24 loader, tokenizer and representation."""\ndef load_owned_base(base_dir, loading, deadline):\n    import json,torch\n    from transformers import Olmo2Config,Olmo2Model\n    original = Olmo2Config.from_pretrained(base_dir, local_files_only=True, trust_remote_code=False)\n    assert (original.model_type, original.hidden_size, original.num_hidden_layers) == (\'olmo2\',4096,32)\n    assert (original.num_attention_heads, original.num_key_value_heads, original.vocab_size) == (32,32,100352)\n    assert original.intermediate_size == 11008 and not original.tie_word_embeddings and not original.attention_bias\n    assert original.bos_token_id is None and original.eos_token_id == 100257 and original.pad_token_id == 100277\n    assert original.rope_parameters[\'rope_theta\'] == 500000 and original.rope_parameters[\'rope_type\'] == \'default\'\n    assert original.rms_norm_eps == 1e-6 and original.attention_dropout == 0 and original.max_position_embeddings == 4096\n    assert not getattr(original,\'quantization_config\',None) and not getattr(original,\'auto_map\',None)\n    # Instantiate only the retained prefix: no full32 GPU model is ever constructed and pruned.\n    retained_config = Olmo2Config.from_dict(original.to_dict())\n    retained_config.num_hidden_layers = 24\n    assert retained_config.hidden_size == 4096 and retained_config.num_hidden_layers == 24\n    index = json.loads((base_dir/\'model.safetensors.index.json\').read_text())[\'weight_map\']\n    assert len(index) == 355 and \'lm_head.weight\' in index\n    original_layer_keys = {key for key in index if key.startswith(\'model.layers.\')}\n    assert len(original_layer_keys) == 352\n    assert {int(key.split(\'.\')[2]) for key in original_layer_keys} == set(range(32))\n    ignored_index_keys = {\'lm_head.weight\'} | {key for key in original_layer_keys if int(key.split(\'.\')[2]) >= 24}\n    used_index_keys = set(index) - ignored_index_keys\n    assert len(ignored_index_keys) == 89 and len(used_index_keys) == 266\n    expected_ignored = {key.removeprefix(\'model.\') for key in ignored_index_keys}\n    expected_used = {key.removeprefix(\'model.\') for key in used_index_keys}\n    def load_base():\n        deadline()\n        base, info = Olmo2Model.from_pretrained(base_dir, config=retained_config, dtype=torch.float16, device_map={"": 0},\n            attn_implementation="eager", trust_remote_code=False, local_files_only=True, output_loading_info=True)\n        assert not info.get("missing_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"), info\n        assert {key.removeprefix(\'model.\') for key in info.get(\'unexpected_keys\', [])} == expected_ignored, info\n        assert set(base.state_dict()) == expected_used\n        assert sum(parameter.numel() for parameter in base.parameters()) == 5268443136\n        assert len(base.layers) == 24 and not hasattr(base, "lm_head")\n        assert base.__class__.__name__ == \'Olmo2Model\' and base.__class__.__module__.startswith(\'transformers.models.olmo2.\')\n        assert base.config._attn_implementation == \'eager\'\n        assert not any(name.endswith(\'lm_head\') for name, module in base.named_modules())\n        assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in base.parameters())\n        assert all(parameter.dtype == torch.float16 for parameter in base.parameters())\n        base.requires_grad_(False)\n        loading.append(info)\n        return base\n    return load_base()\n\ndef make_representation(model,tokenizer,tokens,layers,norms):\n    import torch\n    def representation(rows):\n        batch = tokenizer.pad({"input_ids": [tokens[i] for i in rows]}, return_tensors="pt").to("cuda:0")\n        with torch.autocast("cuda", dtype=torch.float16):\n            hidden = model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hidden) == 25\n        pieces = []\n        with torch.autocast("cuda", enabled=False):\n            for layer, norm in zip(layers, norms):\n                mask = batch.attention_mask.unsqueeze(-1).float()\n                mask[:, 0] = 0\n                assert bool((mask.sum(1) > 0).all())\n                mean = (hidden[layer].float() * mask).sum(1) / mask.sum(1)\n                pieces.append(norm(mean))\n            z = torch.cat(pieces, -1)\n        assert bool(torch.isfinite(z).all())\n        return z, int(batch.attention_mask.sum()), int(batch.input_ids.numel())\n    return representation\n\ndef tokenize_owned(base_dir,texts,tokenizer=None):\n    from transformers import AutoTokenizer,Olmo2Config\n    if tokenizer is None:\n        original = Olmo2Config.from_pretrained(base_dir,local_files_only=True,trust_remote_code=False)\n        tokenizer = AutoTokenizer.from_pretrained(str(base_dir), use_fast=True, local_files_only=True, trust_remote_code=False)\n        assert tokenizer.is_fast and tokenizer.__class__.__module__.startswith(\'transformers.\')\n        # Official tokenizer metadata aliases BOS to EOS; model config has no native BOS.\n        assert tokenizer.bos_token_id == tokenizer.eos_token_id == 100257 and tokenizer.pad_token_id == 100277\n        assert len(tokenizer) == 100278 and original.vocab_size == 100352\n        assert tokenizer.convert_tokens_to_ids(\'<|endoftext|>\') == 100257\n        assert tokenizer.convert_tokens_to_ids(\'<|pad|>\') == 100277\n        tokenizer.padding_side = \'right\'\n    content = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=446)[\'input_ids\']\n    tokens = [[100257]+row+[100257] for row in content]\n    assert all(2 <= len(row) <= 448 and row[0] == row[-1] == 100257 for row in tokens)\n    del content\n    return tokenizer,tokens', 'olmo-spec.json': '{\n  "diagnostic_only_authorized": false,\n  "training_worker_sha256": "b15a93fa754261c1b8e0176684bdc04b1b4692a5c63ee040394b9e93fa8e882d",\n  "inference_authorized": true,\n  "source_hashes": {\n    "olmo_component.py": "32c7f5c4f89bced4e21bc5e852e1f9086b95d10842f009d5799a5c10be875f45",\n    "olmo_owned_path.py": "8e702264a084b93be8dbce9b1c50e59cd9b12ca0ded11ff67a7da73bbc642649"\n  },\n  "core_versions": {\n    "torch": "2.11.0+cu128",\n    "transformers": "5.16.1",\n    "peft": "0.20.0",\n    "accelerate": "1.14.0",\n    "safetensors": "0.8.0",\n    "numpy": "2.1.3",\n    "pandas": "2.3.3"\n  },\n  "base_files": [\n    {\n      "name": "model-00001-of-00006.safetensors",\n      "bytes": 2474775744,\n      "sha256": "215fd4ad7d0f5d83a6bbb2322cc2c76080b1fdac05e438606210442220c4ed9f"\n    },\n    {\n      "name": "model-00002-of-00006.safetensors",\n      "bytes": 2495815296,\n      "sha256": "08575313e1863e7c7af7ff12831033342ad39deea4635545eaf27b65d4518e51"\n    },\n    {\n      "name": "model-00003-of-00006.safetensors",\n      "bytes": 2462277424,\n      "sha256": "8b575321f45f7b7347358c75fb36d10bb3b3422802327c97c66eedea2718ec9d"\n    },\n    {\n      "name": "model-00004-of-00006.safetensors",\n      "bytes": 2428706272,\n      "sha256": "b46c6e7a37da4a188bbff066a4bea020e2576c49fb6c735e6208b33de0f0e76a"\n    },\n    {\n      "name": "model-00005-of-00006.safetensors",\n      "bytes": 2428706272,\n      "sha256": "1b89fd1f9c7d8712a48471bfaf31a26be0e83932ebe3b7d111fffd9aa0560362"\n    },\n    {\n      "name": "model-00006-of-00006.safetensors",\n      "bytes": 2306994712,\n      "sha256": "d131e899a6209c67b8c046bf747ae4e03850179c79f4406e73ced08671c748ff"\n    },\n    {\n      "name": "config.json",\n      "bytes": 623,\n      "sha256": "f1e210de0ba704c579768c154bf57e36e170868ffecb060a6e0d3f2e6b0b9710"\n    },\n    {\n      "name": "conversion-progress.json",\n      "bytes": 88783,\n      "sha256": "5d5856493d980f5cc2371c6e16769cb627174fc5cf09c8419353b4413a5240b9"\n    },\n    {\n      "name": "generation_config.json",\n      "bytes": 126,\n      "sha256": "73e7f06e4ea172c6edeb1a44cab4a603e76962a85ea7ef3245769ba46ca23a30"\n    },\n    {\n      "name": "LICENSE_APACHE_2_0.txt",\n      "bytes": 11358,\n      "sha256": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30"\n    },\n    {\n      "name": "merges.txt",\n      "bytes": 916646,\n      "sha256": "b6fe424e334903f7fb84d3a106d9730455f4744b9fe3c21ee136d97a00e72502"\n    },\n    {\n      "name": "MODEL_WEIGHT_LICENSE_DECLARATION.txt",\n      "bytes": 603,\n      "sha256": "8ff0e78dea979a376700398e2e63467a5f6c39d53079f710897055489ba9faa7"\n    },\n    {\n      "name": "model.safetensors.index.json",\n      "bytes": 29593,\n      "sha256": "76424611ebaa0c3320284c7fb7a8fe8a05355f7ce4d0688097ae40ba634e02a3"\n    },\n    {\n      "name": "NOTICE.md",\n      "bytes": 690,\n      "sha256": "46c3931560a02beb073d6bc7b2d39d03330e289712ab1d8eb94c1e7a0c844201"\n    },\n    {\n      "name": "README.md",\n      "bytes": 12098,\n      "sha256": "a87d5b59422c6040caf0fc62cbcb5f7ace6c71489c1173cf6d9e000777eac962"\n    },\n    {\n      "name": "special_tokens_map.json",\n      "bytes": 125,\n      "sha256": "3c6bf7c09d5473c303cee8575a22bb51e5153c17d177a721b43cd4785c6d09ae"\n    },\n    {\n      "name": "tokenizer_config.json",\n      "bytes": 4345,\n      "sha256": "91c69c665697785ace4ec7dbd159e1839bc9bb5033ab05a56bb0547521dc9ab0"\n    },\n    {\n      "name": "tokenizer.json",\n      "bytes": 7137177,\n      "sha256": "73fd5254624f39a88e3faac6a8e11300fc3c735ed37880d4f4f08db898eaecca"\n    },\n    {\n      "name": "upstream-model.safetensors.index.json",\n      "bytes": 29594,\n      "sha256": "c285ec59423a5a7203724c6258f61d691c0816e4c3f79b1fede776298f894b07"\n    },\n    {\n      "name": "vocab.json",\n      "bytes": 1611056,\n      "sha256": "9e14712c91b37c7aab74b1306baa46ac342d620637a4b44523cdc3aec7d24195"\n    }\n  ],\n  "synthetic_ids": [\n    "synthetic-a",\n    "synthetic-b",\n    "synthetic-c",\n    "synthetic-d",\n    "synthetic-e"\n  ],\n  "classes": [\n    "claude-fable-5",\n    "claude-opus-4-6",\n    "claude-opus-4-8",\n    "claude-opus-5",\n    "claude-sonnet-5",\n    "gemini-3.1-pro-preview",\n    "gemini-3.6-flash",\n    "gemini-3.7-flash",\n    "gemini-3.8-flash",\n    "glm-5.3",\n    "glm-5.3-flash",\n    "gpt-5.5",\n    "gpt-5.6-luna",\n    "gpt-5.6-sol",\n    "gpt-5.6-terra",\n    "gpt-6-astra",\n    "kimi-k3",\n    "qwen3.8-max"\n  ],\n  "base_revision": "7df9a82518afdecae4e8c026b27adccc8c1f0032",\n  "production_admitted": true,\n  "synthetic_texts": [\n    "The quiet observatory opened before sunrise.",\n    "A small program sorts seven coloured stones.",\n    "Please compare the two imaginary calendars carefully."\n  ],\n  "train_sha256": "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707",\n  "base_manifest_sha256": "09fbb318fb7d036162159cc8936e7bd18ce6c24fa8ab192cbfdedaf390045fbe",\n  "diagnostic_cap_seconds": 1200,\n  "no_auto_retry": true,\n  "root_production_admitted": true,\n  "public_asset_release_audited": true,\n  "compact_manifest_sha256": "3ccc36f4a36bc21e09c43a8a32b0d7f6216004cf872835dd41e55925aab51f85",\n  "actual_GPU_diagnostic_audit_sha256": "7e3f981139841f5686648e20bdf365b9da7b5906c1692e8921f9aa813da443bf",\n  "public_file_allowlist": [\n    "adapter/adapter_model.safetensors",\n    "adapter/adapter_config.json",\n    "heads.safetensors",\n    "readouts.npz",\n    "synthetic_anchors.npz",\n    "LICENSE_APACHE_2_0.txt",\n    "NOTICE.md",\n    "provenance.json"\n  ],\n  "changed_public_loader_runtime_audit_sha256": "f7b8168579a8d5bf2ac564a7f86033a176326f90b282192cda9903425a1dfa7a",\n  "base_ref": "lightningv08/olmo2-7b-fp16-offline-weights/1",\n  "public_dataset_ref": "lightningv08/ntl-olmo2-native24-final-inference-assets/2",\n  "public_dataset_id": 12692007\n}\n', 'nishant_0592_original.ipynb': '{"metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"},"language_info":{"name":"python","version":"3.13.15","mimetype":"text/x-python","codemirror_mode":{"name":"ipython","version":3},"pygments_lexer":"ipython3","nbconvert_exporter":"python","file_extension":".py"}},"nbformat_minor":4,"nbformat":4,"cells":[{"id":"c84dbe8d","cell_type":"markdown","source":"# Two-stage LoRA on the public pool (LB 0.592)\\n\\n**The problem in one line.** Telling Claude from GPT is easy. Telling `gemini 3.7 flash` from `gemini 3.8 flash` is the whole game, and the host gives us only 150 passages per model to learn it from.\\n\\n**The idea.** Other Kagglers have published about 35,000 passages from these same 18 models, written to prompts in the competition\'s style. That is 13 times the training set. So we teach in two stages, the way you\'d learn a language:\\n\\n1. **Read a lot first.** Train a LoRA classifier on the public passages only. It learns what each model *sounds like*.\\n2. **Then listen to the teacher.** Fine tune it for 2 epochs on the host\'s 2,699 passages. It learns how each model sounded *when the host asked*: same settings, same formatting.\\n\\nStage 1 alone already matches a LoRA trained on the host data. Stage 2 adds another 9 points.\\n\\n| Model | Local CV (5-fold, grouped by prompt) | Public LB |\\n|---|---|---|\\n| n-gram SVC on train + pool | 0.435 | |\\n| Qwen3-8B LoRA, public passages only (stage 1) | 0.483 | |\\n| Qwen3-8B LoRA, stage 1 + stage 2 | 0.569 | |\\n| Qwen3-14B LoRA, stage 1 + stage 2 | 0.569 | |\\n| **This notebook:** 8B + 14B + n-gram, centred twin, balancing | **0.593** | **0.592** |\\n\\nThe two LoRAs score the same alone but disagree on 22% of passages, which is why blending them helps.\\n\\n**What runs here** (inference only, about 1 h on 2x T4 for the full test):\\n1. An n-gram model, fit here on train + pool.\\n2. Two LoRA checkpoints (Qwen3-8B and Qwen3-14B), trained offline. The training recipe is at the end.\\n3. A \\"centred twin\\" that compares passages answering the same question.\\n4. A log-probability blend, then class balancing.\\n","metadata":{}},{"id":"e1c82b1c","cell_type":"code","source":"import gc, glob, json, os, re, time\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\n\\nT0 = time.time()\\nINPUT_DIR = \\"/kaggle/input\\"\\n\\n# Blend weights, fit on out-of-fold predictions (log probabilities are multiplied by these and summed).\\nW_SVC = 1.021\\nLORAS = {  # checkpoint folder: blend weight, centred-twin weight, batch size, base model to look for\\n    \\"lora8b_v1\\":  dict(w=0.226, twin_w=0.079, bs=16, base=dict(model_type=\\"qwen3\\", hidden_size=4096, num_hidden_layers=36)),\\n    \\"lora14b_v1\\": dict(w=0.429, twin_w=0.150, bs=8,  base=dict(model_type=\\"qwen3\\", hidden_size=5120, num_hidden_layers=40)),\\n}\\nTWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin\\n\\n# Column order of the checkpoints\' heads (checked against each checkpoint\'s meta.json below).\\nLABELS = [\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\\n          \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\\n          \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\',\\n          \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\']\\nK = len(LABELS)\\n\\n\\ndef find(pattern):\\n    hits = sorted(glob.glob(f\\"{INPUT_DIR}/**/{pattern}\\", recursive=True))\\n    if not hits:\\n        raise FileNotFoundError(pattern)\\n    return hits[0]\\n\\n\\ndef softmax(x):\\n    e = np.exp(x - x.max(1, keepdims=True))\\n    return e / e.sum(1, keepdims=True)\\n\\n\\ntrain = pd.read_csv(find(\\"train.csv\\"))\\ntest = pd.read_csv(find(\\"test.csv\\"))\\ntest[\\"text\\"] = test[\\"text\\"].fillna(\\"\\").astype(str)\\ny = train[\\"model\\"].map({l: i for i, l in enumerate(LABELS)}).values\\nLOGIT = np.zeros((len(test), K))   # every member adds w * log p here\\nprint(\\"train\\", train.shape, \\"test\\", test.shape, flush=True)\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"2df31574","cell_type":"markdown","source":"## 1. The n gram model\\n\\nEach model has typing habits. In the training data claude and GLM never use curly quotes, while gpt uses a curly apostrophe in about half its passages. Gemini almost never uses an em dash, while Claude uses one in three passages out of four. Character n grams pick this up.\\n\\nWe look at each passage four ways: raw characters +  words plus punctuation +  the *shape* of the text (`Xxxx xx x`) + function words only, with every content word replaced by `CONTENT` so the topic can\'t leak in. A linear SVM sits on top, trained on train + the public pool with balanced class weights.\\n","metadata":{}},{"id":"e36004a4","cell_type":"code","source":"from scipy.sparse import hstack\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.svm import LinearSVC\\n\\nFUNCTION_WORDS = set(\\"a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little\\".split())\\nVIEWS = [(\\"char\\", 3, 5, 250000, 1.0), (\\"word\\", 1, 3, 180000, 0.7), (\\"shape\\", 2, 6, 60000, 0.5), (\\"function\\", 1, 3, 50000, 0.5)]\\n\\n\\ndef shape_text(t):\\n    return re.sub(r\\"[^\\\\W\\\\d_]+\\", lambda m: \\"X\\" if m[0].isupper() else \\"x\\", t)\\n\\n\\ndef function_text(t):\\n    return \\" \\".join(w.lower() if w.lower() in FUNCTION_WORDS or not w.isalnum() else \\"CONTENT\\"\\n                    for w in re.findall(r\\"\\\\w+|[^\\\\w\\\\s]\\", t))\\n\\n\\npool = pd.read_parquet(find(\\"pool_v1.parquet\\"))\\nfit_texts = train.text.astype(str).tolist() + pool.text.tolist()\\nfit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]\\nA, B = [], []\\nfor view, lo, hi, max_features, weight in VIEWS:\\n    prep = shape_text if view == \\"shape\\" else function_text if view == \\"function\\" else (lambda s: s)\\n    analyzer = \\"char\\" if view == \\"shape\\" else \\"word\\" if view == \\"function\\" else view\\n    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,\\n                max_features=max_features, dtype=np.float32)\\n    if analyzer == \\"word\\":\\n        opts[\\"token_pattern\\"] = r\\"(?u)\\\\b\\\\w+\\\\b|[^\\\\w\\\\s]\\"\\n    vec = TfidfVectorizer(**opts)\\n    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)\\n    B.append(vec.transform([prep(t) for t in test.text]) * weight)\\nsvc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight=\\"balanced\\").fit(hstack(A, format=\\"csr\\"), fit_y)\\nP_svc = softmax(svc.decision_function(hstack(B, format=\\"csr\\")))\\nLOGIT += W_SVC * np.log(np.clip(P_svc, 1e-9, 1))\\ndel A, B, svc, pool\\ngc.collect()\\nprint(f\\"n-gram model done ({time.time() - T0:.0f}s)\\", flush=True)\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"961a630a","cell_type":"markdown","source":"## 2. The LoRA classifiers\\n\\nEach checkpoint is a LoRA adapter on the **first 24 layers of Qwen3 8B** (or the first 27 of Qwen3 14B). We don\'t need the rest: style lives in the early and middle layers, while the last layers mostly encode the topic, which is useless here because test prompts are new.\\n\\nFor each passage we:\\n1. merge the adapter into the weights once (`W <- W + B A * alpha / r`), so no `peft` is needed\\n2. run the passage and take the mean hidden state at 8 depths (layers 2, 4, 6, 8, 12, 16, 20, 24), each through its own LayerNorm;\\n3. concatenate them (8 x 4096 numbers) and apply a linear layer to get 18 scores.\\n\\nThe final RMSNorm is switched off (`Identity`) because the checkpoints were trained that way. On a T4 everything runs in fp16, a batch that overflows is redone in bf16.\\n","metadata":{}},{"id":"4b38467c","cell_type":"code","source":"from safetensors.torch import load_file\\nfrom transformers import AutoConfig, AutoModel, AutoTokenizer\\n\\n\\ndef find_base(spec):\\n    for cfg in sorted(glob.glob(f\\"{INPUT_DIR}/**/config.json\\", recursive=True)):\\n        try:\\n            c = json.load(open(cfg))\\n        except Exception:\\n            continue\\n        if all(c.get(k) == v for k, v in spec.items()):\\n            return os.path.dirname(cfg)\\n    raise FileNotFoundError(f\\"attach the base model {spec}\\")\\n\\n\\ndef load_trunk(base_dir, keep, dtype):\\n    \\"\\"\\"First `keep` decoder layers, split over both T4s when it does not fit on one.\\"\\"\\"\\n    mc = AutoConfig.from_pretrained(base_dir)\\n    mc.num_hidden_layers, mc.use_cache = keep, False\\n    if getattr(mc, \\"layer_types\\", None) is not None:\\n        mc.layer_types = mc.layer_types[:keep]\\n    hd = getattr(mc, \\"head_dim\\", mc.hidden_size // mc.num_attention_heads)\\n    gib = 2 * (mc.vocab_size * mc.hidden_size + keep * (2 * mc.hidden_size * mc.num_attention_heads * hd\\n               + 2 * mc.hidden_size * mc.num_key_value_heads * hd + 3 * mc.hidden_size * mc.intermediate_size)) / 2 ** 30\\n    if gib > 13 and torch.cuda.device_count() > 1:\\n        mem = {i: \\"13GiB\\" for i in range(torch.cuda.device_count())}\\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map=\\"auto\\", max_memory=mem).eval()\\n    else:\\n        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to(\\"cuda:0\\").eval()\\n    model.norm = nn.Identity()\\n    return model\\n\\n\\n@torch.no_grad()\\ndef merge_lora(model, adapter_dir):\\n    acfg = json.load(open(os.path.join(adapter_dir, \\"adapter_config.json\\")))\\n    scale = acfg[\\"lora_alpha\\"] / acfg[\\"r\\"]\\n    sd = load_file(os.path.join(adapter_dir, \\"adapter_model.safetensors\\"))\\n    modules = dict(model.named_modules())\\n    for k, A in sd.items():\\n        if \\".lora_A.\\" in k:\\n            name = k.split(\\".lora_A.\\")[0]\\n            lin = modules[name[name.index(\\"layers.\\"):]]\\n            Bm = sd[k.replace(\\".lora_A.\\", \\".lora_B.\\")]\\n            lin.weight.add_(((Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))\\n\\n\\n@torch.no_grad()\\ndef lora_features(ckpt_dir, base_dir, texts, bs):\\n    \\"\\"\\"Pooled features Z (n, 8 * hidden) and the checkpoint\'s heads.\\"\\"\\"\\n    meta = json.load(open(os.path.join(ckpt_dir, \\"meta.json\\")))\\n    assert meta[\\"labels\\"] == LABELS, \\"label order differs from the checkpoint\'s heads\\"\\n    cfg = meta[\\"cfg\\"]\\n    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\\n    tok = AutoTokenizer.from_pretrained(base_dir)\\n    tok.padding_side = \\"right\\"\\n    if tok.pad_token is None:\\n        tok.pad_token = tok.eos_token\\n    model = load_trunk(base_dir, cfg[\\"keep\\"], dtype)\\n    merge_lora(model, os.path.join(ckpt_dir, \\"adapter\\"))\\n    heads = load_file(os.path.join(ckpt_dir, \\"heads.safetensors\\"))\\n    layers, H = cfg[\\"layers\\"], model.config.hidden_size\\n    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to(\\"cuda:0\\").float()\\n    norms.load_state_dict({k[len(\\"norms.\\"):]: v for k, v in heads.items() if k.startswith(\\"norms.\\")})\\n    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg[\\"max_len\\"])[\\"input_ids\\"]\\n    ids = [x + [tok.eos_token_id] for x in ids]\\n    order = np.argsort([len(x) for x in ids], kind=\\"stable\\")   # similar lengths together = less padding\\n    Z = np.zeros((len(ids), H * len(layers)), np.float32)\\n    for s in range(0, len(order), bs):\\n        idx = order[s:s + bs]\\n        enc = tok.pad({\\"input_ids\\": [ids[i] for i in idx]}, return_tensors=\\"pt\\").to(model.embed_tokens.weight.device)\\n        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\\n        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow: redo this batch in bf16\\n            model.to(torch.bfloat16)\\n            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\\n            model.to(dtype)\\n        m = enc[\\"attention_mask\\"].unsqueeze(-1).float().to(\\"cuda:0\\")\\n        m[:, 0] = 0   # the first token is an attention sink; leave it out of the mean\\n        den = m.sum(1).clamp(min=1)\\n        Z[idx] = torch.cat([nm((hs[l].to(\\"cuda:0\\").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()\\n    del model\\n    torch.cuda.empty_cache()\\n    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith(\\"norms.\\")}\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"a3af9b02","cell_type":"markdown","source":"## 3. Compare answers to the same question\\n\\nPicture two test passages that answer the same prompt. Whatever they share, the topic, is noise for us. Subtract the pair\'s average and what\'s left is mostly *how* each one was written.\\n\\nSo we cluster the test passages by topic (content word TFIDF, average linkage). In each cluster of 2 to 18 passages, we subtract the cluster mean from every passage\'s LoRA features and read them with a second head, which was trained during fine-tuning on exactly such differences. It adds a little evidence on top of the plain head.\\n\\nHow much this helps depends on how many test passages share a prompt, which we don\'t know. In simulation it gives +0.5 points at 1-2 passages per prompt and more with larger groups. With no clusters it simply does nothing.\\n","metadata":{}},{"id":"3b13182f","cell_type":"code","source":"from scipy.cluster.hierarchy import fcluster, linkage\\nfrom scipy.spatial.distance import squareform\\n\\nXt = TfidfVectorizer(stop_words=\\"english\\", sublinear_tf=True).fit_transform(test.text.tolist())\\nD = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)\\nnp.fill_diagonal(D, 0)\\ngroups = fcluster(linkage(squareform(D, checks=False), method=\\"average\\"), TWIN[\\"thresh\\"], criterion=\\"distance\\") \\\\\\n    if len(test) >= 2 else np.ones(len(test), dtype=int)\\nsize = np.bincount(groups)[groups]\\ntwin_rows = (size >= TWIN[\\"min_size\\"]) & (size <= TWIN[\\"max_size\\"])\\ndel D\\nprint(f\\"topic clusters: {int((size == 1).sum())} singletons, {int(twin_rows.sum())} passages in clusters of 2-18\\", flush=True)\\n\\nfor ckpt_name, mc in LORAS.items():\\n    ckpt = os.path.dirname(find(f\\"{ckpt_name}/meta.json\\"))\\n    Z, heads = lora_features(ckpt, find_base(mc[\\"base\\"]), test.text.tolist(), mc[\\"bs\\"])\\n    LOGIT += mc[\\"w\\"] * np.log(np.clip(softmax(Z @ heads[\\"plain.weight\\"].T + heads[\\"plain.bias\\"]), 1e-9, 1))\\n    if twin_rows.any():\\n        Zc = Z.copy()\\n        for g in np.unique(groups[twin_rows]):\\n            Zc[groups == g] -= Zc[groups == g].mean(0)\\n        centred = Zc[twin_rows] @ heads[\\"centered.weight\\"].T + heads[\\"centered.bias\\"]\\n        LOGIT[twin_rows] += mc[\\"twin_w\\"] * np.log(np.clip(softmax(centred), 1e-9, 1))\\n    del Z\\n    gc.collect()\\n    print(f\\"{ckpt_name} done ({time.time() - T0:.0f}s)\\", flush=True)\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"06f3f5f0","cell_type":"markdown","source":"## 4. Blend, then balance\\n\\nThe members are combined by adding their weighted log-probabilities, which amounts to multiplying their probabilities.\\n\\nOne last fix. Our models learned partly from the public pool, where some models have 2,700 passages and others 1,100, so they carry slightly wrong priors. The test set, like train, should hold roughly equal numbers of each model. **Sinkhorn balancing** rescales each class\'s column until every class gets about 1/18 of the probability mass, then renormalises each row, repeated 100 times. It uses no labels. Locally it is worth about +0.5 points, on a test set that is far from balanced it would hurt.\\n","metadata":{}},{"id":"42f60078","cell_type":"code","source":"P = softmax(LOGIT)\\nif len(test) >= 50:   # skip on the 3-row public test.csv\\n    target = np.full(K, len(P) / K)\\n    for _ in range(100):\\n        P *= target / P.sum(0)\\n        P /= P.sum(1, keepdims=True)\\nsub = pd.DataFrame({\\"id\\": test[\\"id\\"].astype(str), \\"model\\": np.array(LABELS)[P.argmax(1)]})\\nsub.to_parquet(\\"submission.parquet\\", index=False)\\nprint(sub[\\"model\\"].value_counts().to_string())\\nprint(f\\"done in {time.time() - T0:.0f}s\\")\\n","metadata":{},"outputs":[],"execution_count":null},{"id":"af1da69e","cell_type":"markdown","source":"## How the checkpoints were trained\\n\\nBoth checkpoints use the same recipe. It is the public \\"LoRA24\\" recipe with one change: a first stage on public data.\\n\\n- **Model:** Qwen3-8B-Base (first 24 layers) or Qwen3-14B-Base (first 27 layers), LoRA rank 16, alpha 32, dropout 0.05 on every linear layer, 8 pooled layers with a LayerNorm each, two linear heads (plain and centred).\\n- **Batches:** triplets of passages that answer the same prompt, 6 triplets per step. The loss is the cross-entropy of the plain head plus the cross-entropy of the centred head on triplet-centred features, which teaches the model to tell apart answers to *the same* question.\\n- **Stage 1:** 1 epoch over the 34,940 public passages, class-weighted cross-entropy (the pool is unbalanced), AdamW lr 1e-4 (LoRA) / 1e-3 (heads), 10% warm-up then cosine.\\n- **Stage 2:** 2 epochs on the host\'s 2,699 passages, lr 5e-5 / 5e-4. More epochs over-fit, and mixing pool passages back in made it worse.\\n- Trained in bf16 on one RTX 5090: about 30 min (8B) and 90 min (14B) per checkpoint.\\n\\n**The pool** merges 14 public datasets, keeping single-paragraph passages of 30-300 words with no markdown. Exact duplicates and anything identical to a training passage are dropped. One more filter: if a model *never* shows a mark in the training data (an em dash for Gemini, curly quotes for Claude), its pool passages containing that mark are dropped. That leaves 34,940 passages for all 18 models (1,100-2,700 per model).\\n\\n**What did not help**, so you can skip it: a larger backbone on its own (14B = 8B at 0.569), seed averaging, extra family \\"gates\\" on top of the blend, Hungarian assignment inside topic clusters, and more epochs in stage 2.\\n\\n## Credits\\n\\nPublic passages (pool), all CC BY 4.0 except where noted:\\n[seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages),\\n[willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest),\\n[lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1),\\n[prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset),\\n[prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1) (CC BY-SA 4.0),\\n[alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations),\\n[haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages),\\n[kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages),\\n[lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations),\\n[abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations),\\n[dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations),\\n[ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations),\\n[keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1),\\n[michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1).\\n\\nIdeas and recipes:\\nthe LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7);\\nthe style-aware n-grams from [yangkuangou](https://www.kaggle.com/yangkuangou)\'s notebook\\nadaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering).\\nModels: Qwen3-8B-Base and Qwen3-14B-Base (Qwen team, Apache 2.0).\\n","metadata":{}}]}', 'LICENSE_APACHE_2_0.txt': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'gaussian_core.py': '"""Shared-prompt Gaussian core by prvsiyan, Apache-2.0.\nExtracted without mathematical changes from The Prompt Is Not the Author,\nv1/script359792979: https://www.kaggle.com/code/prvsiyan/the-prompt-is-not-the-author\n"""\nimport numpy as np\nimport itertools\nK=18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n_MASK = {}\n\nLAM_DIFF = 1e6\n\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n', 'gaussian_integration.txt': 'if ckpt_name == "lora8b_v1":\n        joined_texts = test.text.tolist() + train.text.astype(str).tolist()\n        joined_features, heads = lora_features(ckpt, find_base(mc["base"]), joined_texts, mc["bs"])\n        Z = joined_features[:len(test)]\n        train_features = joined_features[len(test):]\n        gaussian = fit(train_features, y, train["prompt_id"].values,\n            dict(red="wpca", d=128, se_shrink="lw", sp_shrink="lw", lam_floor=1e-4,\n                 n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))\n        gaussian["T1"] = gaussian["Tg"] = 3.0\n        if twin_rows.any():\n            group_labels = np.unique(groups[twin_rows], return_inverse=True)[1]\n            gaussian_logp = scores(Z[twin_rows], group_labels, gaussian)\n            assert np.isfinite(gaussian_logp).all()\n            LOGIT[twin_rows] += 0.10 * gaussian_logp\n            print("Grouped Gaussian8B complete", int(twin_rows.sum()), flush=True)\n        prompt_members = np.flatnonzero(train["prompt_id"].values == train["prompt_id"].values[0])[:2]\n        assert len(prompt_members)==2\n        check = scores(train_features[prompt_members], np.zeros(2,dtype=int), gaussian)\n        assert check.shape==(2,K) and np.isfinite(check).all() and np.allclose(np.exp(check).sum(1),1)\n        del joined_features, joined_texts, train_features, gaussian, check\n    else:\n        Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])\n', 'THIRD_PARTY_NOTICES.md': 'Nishant Dahal: Two-stage LoRA on the public pool, v2/script359887828, Apache2.0 source. Original notebook retained in nishant_0592_original.ipynb.\nprvsiyan: The Prompt Is Not the Author, v1/script359792979, Apache2.0 source. gaussian_core.py contains sixteen mathematical functions with unchanged AST; original source SHA256 a7cf18299d3b6d902dfef2069f7d8f435f95b180cdddfd70ffb99e13478346f2.\nkirilltym: Two-Stage LoRA | Gaussian Readout, v4/script360050107. Method credit for the two-stage8B/grouped Gaussian integration with fixed PCA128/temperature3/weight.10. Gaussian implementation is taken from the licensed prvsiyan source.\nDataset nishantdahal/ntl-two-stage-lora v1, ID12693662: CC BY-SA4.0, https://creativecommons.org/licenses/by-sa/4.0/. Upstream public-pool attribution and share-alike notices remain applicable, including the Smol public warmup subset.\nQwen3-8B-Base and Qwen3-14B-Base: Qwen team, Apache2.0. OLMo-2-1124-7B: Allen Institute for AI, Apache2.0. SmolLM3-3B-Base revision d78a42f79198603e614095753484a04c10c2b940: HuggingFaceTB, Apache2.0.\nChanges: grouped Qwen8B Gaussian integration; native OLMo2 log-pool member; full36 two-stage SmolLM3 routed head plus a separately pre-fitted grouped Gaussian; explicit two-worker GPU orchestration after Qwen exit; Unicode/no-pickle transport. Native OLMo core, native Smol tokenization/pooling helper and original Qwen coefficients/balancing are preserved. Smol final readouts and Gaussian use all official training rows and are not OOF artifacts.\n', 'smol_owned_path.py': '"""Minimal byte-preserved scientific functions extracted from the reviewed owned full-fit path."""\n\n\ndef load_owned_base(base_dir, loading, deadline):\n    import torch\n    from transformers import AutoModel\n    def load_base():\n        deadline()\n        base, info = AutoModel.from_pretrained(base_dir, dtype=torch.float16, device_map={"": 0},\n            attn_implementation="sdpa", trust_remote_code=False, local_files_only=True, output_loading_info=True)\n        assert not info.get("missing_keys") and not info.get("mismatched_keys") and not info.get("error_msgs"), info\n        assert set(info.get("unexpected_keys", [])) <= {"lm_head.weight"}, info\n        assert len(base.layers) == 36 and not hasattr(base, "lm_head")\n        assert all(parameter.device.type == "cuda" and parameter.device.index == 0 for parameter in base.parameters())\n        base.requires_grad_(False)\n        loading.append(info)\n        return base\n    return load_base()\n\n\ndef make_representation(model, tokenizer, tokens, layers, norms):\n    import torch\n    def representation(rows):\n        batch = tokenizer.pad({"input_ids": [tokens[i] for i in rows]}, return_tensors="pt").to("cuda:0")\n        with torch.autocast("cuda", dtype=torch.float16):\n            hidden = model(**batch, output_hidden_states=True, use_cache=False).hidden_states\n        assert len(hidden) == 37\n        pieces = []\n        with torch.autocast("cuda", enabled=False):\n            for layer, norm in zip(layers, norms):\n                mask = batch.attention_mask.unsqueeze(-1).float()\n                mask[:, 0] = 0\n                assert bool((mask.sum(1) > 0).all())\n                mean = (hidden[layer].float() * mask).sum(1) / mask.sum(1)\n                pieces.append(norm(mean))\n            z = torch.cat(pieces, -1)\n        assert bool(torch.isfinite(z).all())\n        return z, int(batch.attention_mask.sum()), int(batch.input_ids.numel())\n    return representation\n\n\ndef tokenize_owned(base_dir, texts, tokenizer=None):\n    if tokenizer is None:\n        from transformers import AutoTokenizer\n        tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True, trust_remote_code=False)\n    assert tokenizer.bos_token_id is None and tokenizer.eos_token_id == 128001\n    tokenizer.padding_side = "right"\n    tokenizer.pad_token_id = 128004\n    tokens = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=447)["input_ids"]\n    tokens = [row + [128001] for row in tokens]\n    assert all(2 <= len(row) <= 448 for row in tokens)\n    return tokenizer, tokens', 'center_readout.py': 'import numpy as np\r\n\r\ndef source_logsoftmax(value):\n    value = value - value.max(1, keepdims=True)\n    return value - np.log(np.exp(value).sum(1, keepdims=True))\n\n\ndef add_logprob(p):\n    assert np.isfinite(p).all() and (p >= 0).all()\n    np.testing.assert_allclose(p.sum(-1), 1, rtol=0, atol=2e-5)\n    return np.log(np.clip(p, 1e-9, 1))\n\n\ndef lora_readouts(Z, ro, pools, recovered, adaptive):\n    """Production FP32 affine and pool-local query centering, never global centering."""\n    assert Z.dtype == np.float32 and Z.shape == (539, 12288)\n    assert ro[\'plain_coef\'].dtype == ro[\'center_coef\'].dtype == np.float32\n    lg = Z @ ro[\'plain_coef\'].T + ro[\'plain_intercept\']\n    lg -= lg.max(1, keepdims=True)\n    p = np.exp(lg)\n    plain = add_logprob(p / p.sum(1, keepdims=True))\n    centered = np.zeros((3, 539, 18), dtype=np.float32)\n    for context in range(3):\n        for pool in np.unique(pools[context]):\n            local = np.flatnonzero((pools[context] == pool) & adaptive[context])\n            if not len(local):\n                continue\n            X = np.asarray(Z[local], dtype=np.float32).copy()\n            groups = recovered[context, local]\n            for group in np.unique(groups):\n                i = groups == group\n                X[i] -= X[i].mean(0)\n            lc = X @ ro[\'center_coef\'].T + ro[\'center_intercept\']\n            centered[context, local] = source_logsoftmax(lc)\n    LC = None\n    if \'claude_coef\' in ro:\n        LC = source_logsoftmax(Z @ ro[\'claude_coef\'].T + ro[\'claude_intercept\'])\n    return plain, centered, LC\n\n\n', 'load_gaussian.py': '"""Read-only, no-pickle loading of the frozen owned final450 Gaussian. No fitting."""\nimport json\nimport numpy as np\n\ndef load_gaussian(path,expected_classes,model_hashes):\n    keys={\'mx\',\'sx\',\'W\',\'m\',\'lam\',\'classes\',\'pool_layers\',\'model_adapter_sha256\',\'heads_sha256\',\'readouts_sha256\',\'T1\',\'Tg\',\'cfg_json\'}\n    with np.load(path,allow_pickle=False) as saved:\n        assert set(saved.files)==keys\n        data={k:saved[k].copy() for k in saved.files}\n    assert all(not v.dtype.hasobject for v in data.values())\n    for key,shape in [(\'mx\',(12288,)),(\'sx\',(12288,)),(\'W\',(12288,128)),(\'m\',(18,128)),(\'lam\',(128,))]:\n        assert data[key].shape==shape and data[key].dtype==np.float64 and np.isfinite(data[key]).all()\n    assert (data[\'sx\']>0).all() and (data[\'lam\']>=1e-4).all()\n    assert data[\'classes\'].shape==(18,) and data[\'classes\'].dtype.kind==\'U\' and len(set(data[\'classes\']))==18\n    np.testing.assert_array_equal(data[\'classes\'],np.asarray(expected_classes,dtype=str))\n    np.testing.assert_array_equal(data[\'pool_layers\'],[6,12,18,24,30,36])\n    for key in [\'model_adapter_sha256\',\'heads_sha256\',\'readouts_sha256\']:\n        assert data[key].shape==() and str(data[key])==model_hashes[key]\n    cfg=json.loads(str(data[\'cfg_json\']))\n    assert cfg==dict(red=\'wpca\',d=128,se_shrink=\'lw\',sp_shrink=\'lw\',lam_floor=1e-4,n_exact=3,inj=True,iters=5,repel=True,sp_scale=1.0)\n    assert float(data[\'T1\'])==float(data[\'Tg\'])==3.0\n    prm={k:data[k] for k in [\'mx\',\'sx\',\'W\',\'m\',\'lam\']}\n    prm.update(cfg=cfg,T1=3.0,Tg=3.0)\n    return prm,data[\'classes\']\n', 'worker_barrier.py': '"""File barrier after native loading/replay and before each owned query forward."""\nimport json,os,time\nfrom pathlib import Path\n\ndef ready_and_wait(name,timeout=900):\n    assert name in (\'olmo\',\'smol\')\n    root=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\n    assert root.is_relative_to(Path(\'/kaggle/temp\').resolve())\n    assert os.environ[\'CUDA_VISIBLE_DEVICES\'].startswith(\'GPU-\')\n    path=root/(name+\'-READY.json\');assert not path.exists()\n    path.write_text(json.dumps(dict(name=name,PID=os.getpid(),UUID=os.environ[\'CUDA_VISIBLE_DEVICES\'])))\n    started=time.monotonic()\n    while not (root/\'OWNED-GO\').exists():\n        assert time.monotonic()-started<timeout,\'Bounded READY/GO wait; no retry\'\n        time.sleep(.1)\n', 'smol_readout.py': '"""Native NumPy FP32 routing plus a frozen FP64 Gaussian; no fitting."""\nimport numpy as np\nfrom center_readout import source_logsoftmax,add_logprob\nfrom gaussian_core import scores\n\ndef smol_score(Z,ro,groups,eligible,prm,gaussian_fn=scores):\n    assert Z.dtype==np.float32 and Z.ndim==2 and Z.shape[1]==12288 and np.isfinite(Z).all()\n    n=len(Z);groups=np.asarray(groups);eligible=np.asarray(eligible)\n    assert groups.shape==eligible.shape==(n,) and groups.dtype.kind in \'iu\' and eligible.dtype==bool\n    counts=np.unique(groups,return_inverse=True,return_counts=True)\n    sizes=counts[2][counts[1]]\n    np.testing.assert_array_equal(eligible,(sizes>=2)&(sizes<=18))\n    for route in (\'plain\',\'center\'):\n        assert ro[route+\'_coef\'].dtype==ro[route+\'_intercept\'].dtype==np.float32\n        assert ro[route+\'_coef\'].shape==(18,12288) and ro[route+\'_intercept\'].shape==(18,)\n    # Exactly the validated CPU plain-head order; clipping belongs ONLY here.\n    lg=Z@ro[\'plain_coef\'].T+ro[\'plain_intercept\']\n    lg-=lg.max(1,keepdims=True)\n    p=np.exp(lg)\n    plain=add_logprob(p/p.sum(1,keepdims=True))\n    routed=plain.copy()\n    local=np.flatnonzero(eligible)\n    if len(local):\n        # One query pool; all members of every eligible original group are present.\n        X=np.asarray(Z[local],dtype=np.float32).copy()\n        local_groups=groups[local]\n        for group in np.unique(local_groups):\n            i=local_groups==group\n            X[i]-=X[i].mean(0)\n        lc=X@ro[\'center_coef\'].T+ro[\'center_intercept\']\n        routed[local]=source_logsoftmax(lc)\n    assert routed.dtype==np.float32 and np.isfinite(routed).all()\n    result=routed.astype(np.float64)\n    if len(local):\n        group_labels=np.unique(groups[local],return_inverse=True)[1]\n        gaussian=gaussian_fn(Z[local],group_labels,prm)\n        assert gaussian.dtype==np.float64 and gaussian.shape==(len(local),18) and np.isfinite(gaussian).all()\n        np.testing.assert_allclose(np.exp(gaussian).sum(1),1,rtol=0,atol=1e-10)\n        result[local]+=0.10*gaussian\n    assert result.dtype==np.float64 and np.isfinite(result).all()\n    # No member renormalization, exponentiation or clipping after Gaussian addition.\n    return result,routed\n', 'smol_component.py': '"""Frozen full36 native SmolLM3 loader and B3 representation; no head fitting."""\nimport hashlib,importlib.metadata,json,time\nfrom pathlib import Path\nimport numpy as np\nfrom numeric_transport import load_numeric\n\ndef sha(path):\n    with Path(path).open(\'rb\') as f:return hashlib.file_digest(f,\'sha256\').hexdigest()\n\nclass SmolFinal:\n    def __init__(self,spec_path,assets,base):\n        import importlib.util\n        finder=importlib.util.find_spec\n        importlib.util.find_spec=lambda n,*a,**kw:None if n.split(\'.\')[0]==\'torchao\' else finder(n,*a,**kw)\n        import torch\n        from torch import nn\n        from peft import PeftModel\n        import peft.import_utils as pi\n        pi.is_torchao_available=lambda *a,**kw:False\n        from safetensors.torch import load_file\n        from scipy.special import softmax\n        from smol_owned_path import load_owned_base\n        self.started=time.monotonic();self.S=json.loads(Path(spec_path).read_text());S=self.S\n        self.assets=Path(assets);self.base=Path(base)\n        assert S[\'public_asset_release_admitted\'] and S[\'asset_ref\'],\'Required public input still closed\'\n        assert {n:importlib.metadata.version(n) for n in S[\'core_versions\']}==S[\'core_versions\']\n        assert torch.cuda.device_count()==1 and torch.cuda.get_device_capability(0)==(7,5)\n        torch.set_num_threads(2)\n        torch.cuda.set_per_process_memory_fraction(13*1024**3/torch.cuda.get_device_properties(0).total_memory,0)\n        manifest=self.assets/\'final_inference_manifest.json\'\n        assert sha(manifest)==S[\'compact_manifest_sha256\']\n        M=json.loads(manifest.read_text())\n        for name,record in M[\'files\'].items():assert sha(self.assets/name)==record[\'sha256\'],name\n        for name,digest in S[\'model_files\'].items():assert sha(self.assets/name)==digest,name\n        for record in S[\'base_files\']:assert sha(self.base/record[\'name\'])==record[\'sha256\']\n        loading=[];frozen=load_owned_base(self.base,loading,self.deadline)\n        self.model=PeftModel.from_pretrained(frozen,self.assets/\'adapter\',is_trainable=False,local_files_only=True).eval()\n        assert not any(x.requires_grad for x in self.model.parameters())\n        assert all(x.dtype==torch.float32 for n,x in self.model.named_parameters() if \'lora_\' in n)\n        self.layers=[6,12,18,24,30,36]\n        self.norms=nn.ModuleList([nn.LayerNorm(2048) for _ in self.layers]).cuda().float()\n        heads=load_file(self.assets/\'heads.safetensors\')\n        self.norms.load_state_dict({k.removeprefix(\'norms.\'):v for k,v in heads.items() if k.startswith(\'norms.\')},strict=True)\n        self.norms.eval();self.norms.requires_grad_(False)\n        self.ro=load_numeric(self.assets/\'readouts.npz\')\n        np.testing.assert_array_equal(self.ro[\'classes\'],np.asarray(S[\'classes\'],dtype=str))\n        for route,prefix in [(\'plain\',\'plain\'),(\'center\',\'centered\')]:\n            np.testing.assert_array_equal(self.ro[route+\'_coef\'],heads[prefix+\'.weight\'].numpy())\n            np.testing.assert_array_equal(self.ro[route+\'_intercept\'],heads[prefix+\'.bias\'].numpy())\n        self.norm_calls=[0]*6\n        def observe(i):\n            def hook(*unused):self.norm_calls[i]+=1\n            return hook\n        self.hooks=[norm.register_forward_hook(observe(i)) for i,norm in enumerate(self.norms)]\n        anchors=load_numeric(self.assets/\'final-production-T4-synthetic-anchors.npz\')\n        assert sha(self.assets/\'final-production-T4-synthetic-anchors.npz\')==S[\'production_anchor_sha256\']\n        assert sha(self.assets/\'synthetic3.json\')==S[\'synthetic_sha256\']==str(anchors[\'synthetic_text_sha256\'])\n        for key,rel in [(\'model_adapter_sha256\',\'adapter/adapter_model.safetensors\'),(\'heads_sha256\',\'heads.safetensors\'),(\'readouts_sha256\',\'readouts.npz\')]:assert str(anchors[key])==S[\'model_files\'][rel]\n        np.testing.assert_array_equal(anchors[\'classes\'],self.ro[\'classes\'])\n        a=self.features(json.loads((self.assets/\'synthetic3.json\').read_text())[\'texts\'],sort=False)\n        for key in [\'features\',\'reloaded_features\']:np.testing.assert_allclose(a,anchors[key],rtol=1e-5,atol=1e-5)\n        drift={}\n        for route in [\'plain\',\'center\']:\n            x=a if route==\'plain\' else a-a.mean(0)\n            b=anchors[\'features\'];y=b if route==\'plain\' else b-b.mean(0)\n            d=float(np.abs(softmax(x@self.ro[route+\'_coef\'].T+self.ro[route+\'_intercept\'],axis=1)-softmax(y@self.ro[route+\'_coef\'].T+self.ro[route+\'_intercept\'],axis=1)).max())\n            assert d<=1e-6;drift[route]=d\n        self.anchor_replay=dict(feature_maxabs=float(np.abs(a-anchors[\'features\']).max()),probability_maxabs=drift,feature_rtol=1e-5,feature_atol=1e-5,probability_atol=1e-6,scope=\'Saved SAME-production-T4 synthetic3; cross-backend tolerances not widened\')\n        assert self.norm_calls==[1]*6,\'Exactly one trained LayerNorm per pool, no second LN\'\n\n    def deadline(self):assert time.monotonic()-self.started<9000,\'Bounded Smol worker; no retry\'\n\n    def features(self,texts,sort=True):\n        import torch\n        from smol_owned_path import tokenize_owned,make_representation\n        self.deadline();tokenizer,tokens=tokenize_owned(self.base,texts)\n        fn=make_representation(self.model,tokenizer,tokens,self.layers,self.norms)\n        order=np.argsort([len(t) for t in tokens],kind=\'stable\') if sort else np.arange(len(tokens))\n        z=np.empty((len(tokens),12288),np.float32)\n        with torch.inference_mode():\n            for start in range(0,len(order),3):\n                self.deadline();ii=order[start:start+3];z[ii]=fn(ii)[0].cpu().numpy()\n        assert np.isfinite(z).all();return z\n', 'compute_smol.py': '"""One frozen Smol query forward and read-only Gaussian score, GPU1 only."""\nimport json,os,time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom smol_component import SmolFinal,sha\nfrom numeric_transport import load_numeric,save_numeric\nfrom load_gaussian import load_gaussian\nfrom smol_readout import smol_score\nfrom worker_barrier import ready_and_wait\nHERE=Path(__file__).resolve().parent\nC=json.loads((HERE/\'candidate-config.json\').read_text());S=json.loads((HERE/\'smol-spec.json\').read_text())\nassert C[\'source_ready\'] and C[\'Smol_raw_coefficient\']==.03 and C[\'Smol_Gaussian_weight\']==.10\nassert S[\'public_asset_release_admitted\'] and S[\'asset_ref\']\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve();assert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nassert not (cache/\'smol_score.npz\').exists(),\'No automatic repeat/reuse\'\nassets=[p for p in Path(\'/kaggle/input\').rglob(\'final_inference_manifest.json\') if sha(p)==S[\'compact_manifest_sha256\']]\nmarkers=[p for p in Path(\'/kaggle/input\').rglob(\'smollm3_staging_complete.json\') if sha(p)==S[\'base_manifest_sha256\']]\ntests=[p/\'test.csv\' for p in (Path(\'/kaggle/input/competitions/name-that-llm\'),Path(\'/kaggle/input/name-that-llm\')) if (p/\'test.csv\').is_file()]\nif os.environ.get(\'NTL_QUERY_CSV\'):tests=[Path(os.environ[\'NTL_QUERY_CSV\'])]\nassert len(assets)==len(markers)==len(tests)==1\nquery=pd.read_csv(tests[0],usecols=[\'id\',\'text\'],dtype=str);query[\'text\']=query[\'text\'].fillna(\'\').astype(str)\nids=query.id.to_numpy(dtype=str);groups=load_numeric(cache/\'baseline-groups.npz\')\nnp.testing.assert_array_equal(groups[\'ids\'],ids)\nassert len(set(ids))==len(ids)>0 and os.environ[\'CUDA_VISIBLE_DEVICES\'].startswith(\'GPU-\')\nmodel=SmolFinal(HERE/\'smol-spec.json\',assets[0].parent,markers[0].parent/\'smollm3-base\')\nassert sha(assets[0].parent/\'gaussian-parameters.npz\')==S[\'Gaussian_parameter_sha256\']\nassert sha(assets[0].parent/\'gaussian-metadata.json\')==S[\'Gaussian_metadata_sha256\']\nhashes={key:S[\'model_files\'][rel] for key,rel in [(\'model_adapter_sha256\',\'adapter/adapter_model.safetensors\'),(\'heads_sha256\',\'heads.safetensors\'),(\'readouts_sha256\',\'readouts.npz\')]}\nprm,labels=load_gaussian(assets[0].parent/\'gaussian-parameters.npz\',model.ro[\'classes\'],hashes)\nready_and_wait(\'smol\')\nimport torch\nbegin,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)\nbefore=time.monotonic();begin.record();z=model.features(query.text.tolist());end.record();torch.cuda.synchronize()\nevent_ms=float(begin.elapsed_time(end));assert event_ms>0\nexpected_batches=1+(len(ids)+2)//3\nassert model.norm_calls==[expected_batches]*6,\'One trained LN per pool per batch, including synthetic3 replay\'\nscore,routed=smol_score(z,model.ro,groups[\'groups\'],groups[\'eligible\'],prm)\nsave_numeric(cache/\'smol_score.npz\',ids=ids,labels=labels,score=score)\nsave_numeric(Path(\'/kaggle/working/smol_score_recovery.npz\'),ids=ids,labels=labels,score=score)\nassert torch.cuda.max_memory_reserved()<=13*1024**3\nreport=dict(status=\'FROZEN_SMOL36_ROUTED_GAUSSIAN_QUERY_COMPLETE\',PID=os.getpid(),CUDA_VISIBLE_DEVICES=os.environ[\'CUDA_VISIBLE_DEVICES\'],rows=len(ids),eligible_rows=int(groups[\'eligible\'].sum()),feature_width=12288,feature_dtype=\'float32\',routed_dtype=str(routed.dtype),score_dtype=str(score.dtype),query_cuda_event_ms=event_ms,wall_seconds=time.monotonic()-before,peak_reserved_bytes=torch.cuda.max_memory_reserved(),anchor_replay=model.anchor_replay,norm_calls=model.norm_calls,model_fits=0,head_fits=0,Gaussian_fits=0,raw_member_weight=.03,Gaussian_weight=.10,member_renormalized=False,original_Nishant_groups=True)\nPath(\'/kaggle/working/smol_inference_audit.json\').write_text(json.dumps(report,indent=2))\n', 'smol-spec.json': '{\n  "public_asset_release_admitted": true,\n  "asset_ref": "lightningv08/ntl-olmo2-native24-final-inference-assets/2",\n  "compact_manifest_sha256": "00bf6f993dbc034051f5cd103f1decf50c784c09f7d1bf131738567f0eaf04dc",\n  "base_manifest_sha256": "120112bd002cde8bd7705514467081db4d78646c222b3958b6e3ab0d09098911",\n  "base_revision": "d78a42f79198603e614095753484a04c10c2b940",\n  "base_files": [\n    {\n      "name": "README.md",\n      "bytes": 12340,\n      "sha256": "9bb1fff003df8f9497d2589c345b80817af6864a39938c14edc5207bc5717009",\n      "pinned_git_blob": "48b8618e0d20e47e826a0a842c78a2dc4616d6a9",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/README.md?download=true"\n    },\n    {\n      "name": "config.json",\n      "bytes": 943,\n      "sha256": "e8336e843aecb733691a043ad4209168e1730158b5e03c2672591cc946036a47",\n      "pinned_git_blob": "975eb9a0acea3b739ecdbc4a4f615d688b27b407",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/config.json?download=true"\n    },\n    {\n      "name": "generation_config.json",\n      "bytes": 126,\n      "sha256": "7d830c1a274c4484a50eb34ed68cdbd3611a4981fd4d539c7ec1fed67e01cec4",\n      "pinned_git_blob": "e0b85d8040e51ed5efddf6a2501c3a462b40e6da",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/generation_config.json?download=true"\n    },\n    {\n      "name": "model-00001-of-00002.safetensors",\n      "bytes": 4966315264,\n      "sha256": "7e270ac568ee1880ddbadad66ccdcd9906d52415e8904e2f300c75250b9c7d49",\n      "pinned_git_blob": "81dcd69d7d543ca2f74efcb3750ee947231ed183",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/model-00001-of-00002.safetensors?download=true"\n    },\n    {\n      "name": "model-00002-of-00002.safetensors",\n      "bytes": 1183919744,\n      "sha256": "c6a6e7690a66dcc386a6a9b456686e7c308d45cba884d116c928dafa7fa987ae",\n      "pinned_git_blob": "0d3daab6805b3cd55fc1dd88f5f1d90bc66c183b",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/model-00002-of-00002.safetensors?download=true"\n    },\n    {\n      "name": "model.safetensors.index.json",\n      "bytes": 26899,\n      "sha256": "e3a7254c086c4a78f95ad5864f435d165ffed1ed8998326ce5107b4423c15b76",\n      "pinned_git_blob": "ee6c028c012814d385be51c1d537c345c9fa9457",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/model.safetensors.index.json?download=true"\n    },\n    {\n      "name": "special_tokens_map.json",\n      "bytes": 151,\n      "sha256": "9faaa15579bbf433176493a621142ff580ae2d13d81bf4702cb72f4d6902dc62",\n      "pinned_git_blob": "d2a1bbca9df5663dbd63c80560824f8aef02cf9e",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/special_tokens_map.json?download=true"\n    },\n    {\n      "name": "tokenizer.json",\n      "bytes": 17208861,\n      "sha256": "ab4da6b2aa68247e9c0fa9b97fc7fcc796505038d01f7e144522a65ce0dbd2e5",\n      "pinned_git_blob": "0cd8b7fa7c1f17845e0d5e74f9d5afe90ca3105c",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/tokenizer.json?download=true"\n    },\n    {\n      "name": "tokenizer_config.json",\n      "bytes": 50405,\n      "sha256": "f398d9edb0184aac1c9fd1b1bdcd3a5a1527ae560fd847719635a66e2b54e9cd",\n      "pinned_git_blob": "c57d419f2a96470776f99fa37a9143363d3ae047",\n      "source": "https://huggingface.co/HuggingFaceTB/SmolLM3-3B-Base/resolve/d78a42f79198603e614095753484a04c10c2b940/tokenizer_config.json?download=true"\n    }\n  ],\n  "model_files": {\n    "adapter/adapter_config.json": "bb6ce73b7ee61ef82302407db1fe32fe35d25ddbf1b3867c9f15f04496328780",\n    "adapter/adapter_model.safetensors": "8055ab4ebc2e978d636f5c3c04250a21ea2b084e76f8f5b8756af72a7aa0360a",\n    "heads.safetensors": "776be9a08eaefa409ae5fd7625490a5610aba8df6ad9bcf643c56c898b08e48a",\n    "readouts.npz": "26502c36f8760460ab3f5960632c8f8083d03c77d12eb379c1146f5e59bd6dbc"\n  },\n  "classes": [\n    "claude-fable-5",\n    "claude-opus-4-6",\n    "claude-opus-4-8",\n    "claude-opus-5",\n    "claude-sonnet-5",\n    "gemini-3.1-pro-preview",\n    "gemini-3.6-flash",\n    "gemini-3.7-flash",\n    "gemini-3.8-flash",\n    "glm-5.3",\n    "glm-5.3-flash",\n    "gpt-5.5",\n    "gpt-5.6-luna",\n    "gpt-5.6-sol",\n    "gpt-5.6-terra",\n    "gpt-6-astra",\n    "kimi-k3",\n    "qwen3.8-max"\n  ],\n  "pool_layers": [\n    6,\n    12,\n    18,\n    24,\n    30,\n    36\n  ],\n  "native_batch_size": 3,\n  "feature_width": 12288,\n  "raw_member_weight": 0.03,\n  "Gaussian_weight": 0.1,\n  "eligible_group_sizes": [\n    2,\n    18\n  ],\n  "group_source": "Unmodified Nishant TWIN groups thresh=.85/min_size=2/max_size=18 exported from actual baseline env",\n  "Gaussian_parameter_sha256": "42e18bc621de2690b4c96b4cc5b58f2fc2ee83f486f1923224fd904a27a6a61f",\n  "Gaussian_metadata_sha256": "180cbe047ab0610a4c5d8ece17bae900dc8009bbd9fb4847b8e413e15901d0a6",\n  "production_anchor_sha256": "2969b6138c5d1718d9bf0724a465ef655168038e1885546808e97c26b047230e",\n  "synthetic_sha256": "557e72e1cae134450a834af367e05260e1a90762beccf9e16ae7b74d7ab7fbf9",\n  "core_versions": {\n    "torch": "2.11.0+cu128",\n    "transformers": "5.16.1",\n    "peft": "0.20.0",\n    "accelerate": "1.14.0",\n    "safetensors": "0.8.0",\n    "numpy": "2.1.3",\n    "pandas": "2.3.3"\n  },\n  "memory_GiB": 13,\n  "fit_count_at_runtime": 0,\n  "new_candidate_score": null,\n  "full_current_stack_OOF_available": false\n}\n', 'concurrency_evidence.py': '"""Same native QA requirements at every size; honest short-workload overlap scope."""\nimport math\n\ndef assess_owned_evidence(rows,seen,overlap,ready,audits,pids,mapping):\n    assert isinstance(rows,int) and rows>0\n    assert len(ready)==len(audits)==len(pids)==len(mapping)==2\n    assert len(set(pids))==2 and len({m[\'uuid\'] for m in mapping})==2\n    for i,(r,a) in enumerate(zip(ready,audits)):\n        assert r[\'PID\']==pids[i] and r[\'UUID\']==mapping[i][\'uuid\']\n        assert a[\'CUDA_VISIBLE_DEVICES\']==mapping[i][\'uuid\'] and a[\'rows\']==rows\n        assert math.isfinite(a[\'query_cuda_event_ms\']) and a[\'query_cuda_event_ms\']>0\n        assert 0<a[\'peak_reserved_bytes\']<=13*1024**3\n        native=a[\'synthetic_anchor_replay\'] if i==0 else a[\'anchor_replay\']\n        assert native[\'feature_rtol\']==native[\'feature_atol\']==1e-5 and native[\'probability_atol\']==1e-6\n        assert math.isfinite(native[\'feature_maxabs\']) and native[\'feature_maxabs\']>=0\n        assert set(native[\'probability_maxabs\'])=={\'plain\',\'center\'}\n        assert all(math.isfinite(v) and 0<=v<=1e-6 for v in native[\'probability_maxabs\'].values())\n    assert overlap>=0 and seen<={0,1}\n    proven=seen=={0,1} and overlap>0\n    if rows>=50:\n        assert proven,\'N>=50 requires both sampled actual PID/UUID bindings and simultaneous GPU utilization after GO\'\n    return dict(rows=rows,concurrent_execution_proven=proven,overlap_evidence=\'OBSERVED_PID_UUID_UTIL_OVERLAP\' if proven else \'INSUFFICIENT_SHORT_WORKLOAD\',actual_PID_UUID_util_overlap_samples=overlap,actual_PID_UUID_seen=sorted(seen),both_native_QA_and_positive_query_CUDA_events=True,distinct_worker_PID_UUID_READY_bindings=True,strict_overlap_required=rows>=50)\n', 'run_synthetic_baseline.py': '"""Diagnostic stub only: numeric zeros and fixed synthetic groups, NO baseline model."""\nimport hashlib,json,os\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom numeric_transport import save_numeric\nHERE=Path(__file__).resolve().parent\nassert os.environ.get(\'NTL_RUN_MODE\')==\'diagnostic\'\nC=json.loads((HERE/\'candidate-config.json\').read_text())\nassert C[\'cloud_probe_authorized\'] and not C[\'candidate_admitted\']\ncache=Path(os.environ[\'NTL_CANDIDATE_TEMP\']).resolve()\nassert cache.is_relative_to(Path(\'/kaggle/temp\').resolve())\nfixture=json.loads((HERE/\'diagnostic60.json\').read_text())\nassert fixture[\'rows\']==60 and fixture[\'eligible_rows\']==27\nids=np.asarray(fixture[\'ids\'],dtype=str);groups=np.asarray(fixture[\'groups\'],dtype=np.int64)\nassert len(ids)==len(set(ids))==60 and len(groups)==60\nsize=np.bincount(groups)[groups];eligible=(size>=2)&(size<=18)\nassert int(eligible.sum())==27\nlabels=np.asarray(fixture[\'Nishant_original_labels\'],dtype=str)\nassert len(set(labels))==18 and labels.tolist()!=sorted(labels.tolist())\nquery=Path(os.environ[\'NTL_QUERY_CSV\']);assert query.parent==cache\npd.DataFrame({\'id\':ids,\'text\':fixture[\'texts\']}).to_csv(query,index=False)\ncheck=pd.read_csv(query,dtype=str);np.testing.assert_array_equal(check.id.to_numpy(dtype=str),ids)\nsave_numeric(cache/\'baseline-state.npz\',ids=ids,labels=labels,raw=np.zeros((60,18),np.float64))\nsave_numeric(cache/\'baseline-groups.npz\',ids=ids,groups=groups,size=size,eligible=eligible)\nPath(\'/kaggle/working/diagnostic_stub_audit.json\').write_text(json.dumps(dict(status=\'SYNTHETIC_ZERO_RAW_NOT_BASELINE_PREDICTIONS\',rows=60,eligible_rows=27,group_sizes=[18,4,3,2,19]+[1]*14,fixture_sha256=hashlib.sha256((HERE/\'diagnostic60.json\').read_bytes()).hexdigest(),real_Qwen_or_SVC_or_Gaussian_fit=False,official_rows=0,not_validation=True),indent=2))\n', 'diagnostic60.json': '{\n  "origin": "Own public synthetic prose; this is a runtime diagnostic, not validation",\n  "rows": 60,\n  "eligible_rows": 27,\n  "ids": [\n    "000000-Ж-é-synthetic",\n    "000001-Ж-é-synthetic",\n    "000002-Ж-é-synthetic",\n    "000003-Ж-é-synthetic",\n    "000004-Ж-é-synthetic",\n    "000005-Ж-é-synthetic",\n    "000006-Ж-é-synthetic",\n    "000007-Ж-é-synthetic",\n    "000008-Ж-é-synthetic",\n    "000009-Ж-é-synthetic",\n    "000010-Ж-é-synthetic",\n    "000011-Ж-é-synthetic",\n    "000012-Ж-é-synthetic",\n    "000013-Ж-é-synthetic",\n    "000014-Ж-é-synthetic",\n    "000015-Ж-é-synthetic",\n    "000016-Ж-é-synthetic",\n    "000017-Ж-é-synthetic",\n    "000018-Ж-é-synthetic",\n    "000019-Ж-é-synthetic",\n    "000020-Ж-é-synthetic",\n    "000021-Ж-é-synthetic",\n    "000022-Ж-é-synthetic",\n    "000023-Ж-é-synthetic",\n    "000024-Ж-é-synthetic",\n    "000025-Ж-é-synthetic",\n    "000026-Ж-é-synthetic",\n    "000027-Ж-é-synthetic",\n    "000028-Ж-é-synthetic",\n    "000029-Ж-é-synthetic",\n    "000030-Ж-é-synthetic",\n    "000031-Ж-é-synthetic",\n    "000032-Ж-é-synthetic",\n    "000033-Ж-é-synthetic",\n    "000034-Ж-é-synthetic",\n    "000035-Ж-é-synthetic",\n    "000036-Ж-é-synthetic",\n    "000037-Ж-é-synthetic",\n    "000038-Ж-é-synthetic",\n    "000039-Ж-é-synthetic",\n    "000040-Ж-é-synthetic",\n    "000041-Ж-é-synthetic",\n    "000042-Ж-é-synthetic",\n    "000043-Ж-é-synthetic",\n    "000044-Ж-é-synthetic",\n    "000045-Ж-é-synthetic",\n    "000046-Ж-é-synthetic",\n    "000047-Ж-é-synthetic",\n    "000048-Ж-é-synthetic",\n    "000049-Ж-é-synthetic",\n    "000050-Ж-é-synthetic",\n    "000051-Ж-é-synthetic",\n    "000052-Ж-é-synthetic",\n    "000053-Ж-é-synthetic",\n    "000054-Ж-é-synthetic",\n    "000055-Ж-é-synthetic",\n    "000056-Ж-é-synthetic",\n    "000057-Ж-é-synthetic",\n    "000058-Ж-é-synthetic",\n    "000059-Ж-é-synthetic"\n  ],\n  "texts": [\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 003.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 004.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 005.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 006.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 007.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 008.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 009.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 010.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 011.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 012.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 013.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 014.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 015.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 016.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 017.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 018.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 019.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 020.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 021.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 022.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 023.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 024.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 025.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 026.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 027.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 028.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 029.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 030.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 031.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 032.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 033.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 034.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 035.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 036.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 037.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 038.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 039.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 040.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 041.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 042.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 043.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 044.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 045.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 046.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 047.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 048.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 049.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 050.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 051.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 052.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 053.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 054.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 055.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 056.",\n    "A small copper key rested beside the window. Three quiet taps ended the conversation.\\nSynthetic variant 057.",\n    "The note read: \\"Wait—why?\\" Then came 17 blue marks, two commas, and a final full stop.\\nA new paragraph answered nothing.\\nSynthetic variant 058.",\n    "Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass. Consider the following small experiment: a lantern casts a narrow circle across the desk; outside, rain moves steadily past the glass.\\nSynthetic variant 059."\n  ],\n  "groups": [\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    2,\n    2,\n    2,\n    2,\n    3,\n    3,\n    3,\n    4,\n    4,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    5,\n    6,\n    7,\n    8,\n    9,\n    10,\n    11,\n    12,\n    13,\n    14,\n    15,\n    16,\n    17,\n    18,\n    19\n  ],\n  "Nishant_original_labels": [\n    "gpt-5.5",\n    "gpt-5.6-luna",\n    "gpt-5.6-sol",\n    "gpt-5.6-terra",\n    "gpt-6-astra",\n    "gemini-3.1-pro-preview",\n    "gemini-3.6-flash",\n    "gemini-3.7-flash",\n    "gemini-3.8-flash",\n    "claude-opus-4-6",\n    "claude-opus-4-8",\n    "claude-opus-5",\n    "claude-sonnet-5",\n    "claude-fable-5",\n    "qwen3.8-max",\n    "glm-5.3",\n    "glm-5.3-flash",\n    "kimi-k3"\n  ],\n  "baseline_RAW": "Explicit all-zero FP64[60,18] stub, not real Qwen/SVC/Gaussian predictions",\n  "group_sizes": [\n    18,\n    4,\n    3,\n    2,\n    19,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1,\n    1\n  ]\n}\n', 'candidate-config.json': '{\n  "source_ready": true,\n  "cloud_probe_authorized": false,\n  "candidate_admitted": true,\n  "probe_rows": 5,\n  "OLMo_plain_raw_coefficient": 0.03,\n  "baseline_notebook_id": 139520703,\n  "baseline_version": 2,\n  "baseline_script_version_id": 359887828,\n  "new_baseline_additivity_evidence_admitted": false,\n  "source_sha256": {\n    "baseline_source.py": "8ddd2646d2d263a4e6039b470763f99386aa674133b03b6e991f3095466f2a19",\n    "numeric_transport.py": "0c06ff559a0320e6e52cb4620274db10ff4a599899145d87b459c76ecdaa1b81",\n    "run_baseline.py": "7933ee9bcc150a5ce4ebe67c19a9d66d0ca2f67e405d38a0415b8fb97a553e7e",\n    "run_baseline_with_groups.py": "ec04292f54ba321d71fd0656abbfeed5c1aa97f7cdf7bc44dab4ca411cf1aa00",\n    "compute_olmo.py": "f0cbad0f32ad17c6b3c723addbcf64027651ac414c7e81775b596ca56eca8198",\n    "finish_prediction.py": "9516a12bef87281ecf34841262c5d613f011bf785e3df41a8b03a5a960fd34d5",\n    "run_candidate.py": "a3c20ae5ca1ca4c45959a360fafaef0a63c3d686687a8c551a80110ba609707f",\n    "olmo_component.py": "32c7f5c4f89bced4e21bc5e852e1f9086b95d10842f009d5799a5c10be875f45",\n    "olmo_owned_path.py": "8e702264a084b93be8dbce9b1c50e59cd9b12ca0ded11ff67a7da73bbc642649",\n    "olmo-spec.json": "0f7bc46bb9be6879a53baa829e97fc935e92493f349380fb4b1a9d7c1b8ddd48",\n    "nishant_0592_original.ipynb": "12e1e65bbc48a2839842d439fd7ae3846b9bf0d9913c4cdc308ebc698974b3fb",\n    "LICENSE_APACHE_2_0.txt": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30",\n    "gaussian_core.py": "4b5a1f0c17df93e31ae424ae176135c2292bdd20f70088fd4a1eee76335f3020",\n    "gaussian_integration.txt": "e55275b41e429c372417c4da97089942e079031478e566dcf9d410c61702617a",\n    "THIRD_PARTY_NOTICES.md": "2eb14661be1b35e99f3640a63e536e47efe515ecccfe64cf609d5329ee22d6ce",\n    "smol_owned_path.py": "20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620",\n    "center_readout.py": "c66c5a4b61952c0612b95e601ebc043823ed229d44f1533b0e1930b2e257c3f4",\n    "load_gaussian.py": "5df865c2ce79c6423d8bf55aac0a896261420bf17457c18e3a1cdcf997aeab55",\n    "worker_barrier.py": "c01a5271e0b9bc296810d47309cd12a2791d4741839d15534b677d978119158e",\n    "smol_readout.py": "add20d61c93c891ea828aedf3cbf9509cedca9618ba3eee85cff9386f8e60ad0",\n    "smol_component.py": "e9e8a1f2451f36b7ad101e22b6e43670a866ab0b4b0b791b9ba4823500723490",\n    "compute_smol.py": "203d66c4e98daaed82bd59e4f34d5cea667c95874568605564d0050148bd8c1a",\n    "smol-spec.json": "41292f306a20fdd0630b80540462ca8c0d7a0ef52f9a4ae4ff57819cc4c2c86d",\n    "concurrency_evidence.py": "dc24a2a973df4f9395961443fa56fcdd34217771cf57097dc3644317700ef5d7",\n    "run_synthetic_baseline.py": "236cc4130b3249db111a5faabd8cfed648cc6efc847118919f6213a3bcf98f0c",\n    "diagnostic60.json": "99155edca163146e306f27ad251cb82d645df802cffe432db4376c8f248404c1"\n  },\n  "Gaussian_weight": 0.1,\n  "Gaussian_dimension": 128,\n  "Gaussian_temperature": 3.0,\n  "Gaussian_group_sizes": [\n    2,\n    18\n  ],\n  "Gaussian_method_notebook_id": 138904218,\n  "Gaussian_method_version": 4,\n  "Gaussian_method_script_version_id": 360050107,\n  "Gaussian_core_author": "prvsiyan",\n  "Gaussian_core_source_script_version_id": 359792979,\n  "Gaussian_core_source_license": "Apache-2.0",\n  "Gaussian_core_source_license_evidence": "research/public-source-watch/prvsiyan-0569-ui-binding.json",\n  "Kirill_notebook_code_copied": false,\n  "prior_own_v13_score": 0.594,\n  "new_candidate_score": null,\n  "listing_title_change_authorized": false,\n  "target_numeric_notebook_id": null,\n  "target_ref": "lightningv08/two-stage-gaussian-olmo2-smollm3",\n  "listing_title": "Two-Stage LoRA + OLMo2 + SmolLM3",\n  "Smol_raw_coefficient": 0.03,\n  "Smol_Gaussian_weight": 0.1,\n  "Smol_asset_publication_admitted": true,\n  "parent_v14_notebook_sha256": "3208e88abfc0ee1c1798a48ade2653262918dd0899c8dc64245523cb2fa44866"\n}'}
HASHES={'baseline_source.py': '77cf7bc9e8628d1a609206c060ba12e6d3660a8662671e3316c62421a15b600b', 'numeric_transport.py': '0c06ff559a0320e6e52cb4620274db10ff4a599899145d87b459c76ecdaa1b81', 'run_baseline.py': '7933ee9bcc150a5ce4ebe67c19a9d66d0ca2f67e405d38a0415b8fb97a553e7e', 'run_baseline_with_groups.py': 'ec04292f54ba321d71fd0656abbfeed5c1aa97f7cdf7bc44dab4ca411cf1aa00', 'compute_olmo.py': 'f0cbad0f32ad17c6b3c723addbcf64027651ac414c7e81775b596ca56eca8198', 'finish_prediction.py': '9516a12bef87281ecf34841262c5d613f011bf785e3df41a8b03a5a960fd34d5', 'run_candidate.py': 'a3c20ae5ca1ca4c45959a360fafaef0a63c3d686687a8c551a80110ba609707f', 'olmo_component.py': '32c7f5c4f89bced4e21bc5e852e1f9086b95d10842f009d5799a5c10be875f45', 'olmo_owned_path.py': '8e702264a084b93be8dbce9b1c50e59cd9b12ca0ded11ff67a7da73bbc642649', 'olmo-spec.json': '0f7bc46bb9be6879a53baa829e97fc935e92493f349380fb4b1a9d7c1b8ddd48', 'nishant_0592_original.ipynb': '12e1e65bbc48a2839842d439fd7ae3846b9bf0d9913c4cdc308ebc698974b3fb', 'LICENSE_APACHE_2_0.txt': 'cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30', 'gaussian_core.py': '4b5a1f0c17df93e31ae424ae176135c2292bdd20f70088fd4a1eee76335f3020', 'gaussian_integration.txt': 'e55275b41e429c372417c4da97089942e079031478e566dcf9d410c61702617a', 'THIRD_PARTY_NOTICES.md': '2eb14661be1b35e99f3640a63e536e47efe515ecccfe64cf609d5329ee22d6ce', 'smol_owned_path.py': '20dabafd3f423196863ce3d1fbd4caa8b196bd517a88422561615db003d1e620', 'center_readout.py': 'c66c5a4b61952c0612b95e601ebc043823ed229d44f1533b0e1930b2e257c3f4', 'load_gaussian.py': '5df865c2ce79c6423d8bf55aac0a896261420bf17457c18e3a1cdcf997aeab55', 'worker_barrier.py': 'c01a5271e0b9bc296810d47309cd12a2791d4741839d15534b677d978119158e', 'smol_readout.py': 'add20d61c93c891ea828aedf3cbf9509cedca9618ba3eee85cff9386f8e60ad0', 'smol_component.py': 'e9e8a1f2451f36b7ad101e22b6e43670a866ab0b4b0b791b9ba4823500723490', 'compute_smol.py': '203d66c4e98daaed82bd59e4f34d5cea667c95874568605564d0050148bd8c1a', 'smol-spec.json': '41292f306a20fdd0630b80540462ca8c0d7a0ef52f9a4ae4ff57819cc4c2c86d', 'concurrency_evidence.py': 'dc24a2a973df4f9395961443fa56fcdd34217771cf57097dc3644317700ef5d7', 'run_synthetic_baseline.py': '236cc4130b3249db111a5faabd8cfed648cc6efc847118919f6213a3bcf98f0c', 'diagnostic60.json': '99155edca163146e306f27ad251cb82d645df802cffe432db4376c8f248404c1', 'candidate-config.json': '5c7b39396b937066889f5f13484df992b75527d364315fd145821473d3af4e2d'}
for name,text in FILES.items():
    assert Path(name).name==name
    data=text.encode('utf-8')
    assert hashlib.sha256(data).hexdigest()==HASHES[name]
    (ROOT/name).write_bytes(data)


In [ ]:
import os,subprocess,sys
os.environ['NTL_RUN_MODE']='candidate'
subprocess.run([sys.executable,'-u',str(ROOT/'run_candidate.py')],check=True)
